# S1 Code. Analysis pipeline for discharge disposition prediction after acute stroke

This supplementary notebook reproduces the dataset preparation, descriptive analyses, exploratory PCA, final MLP model development, calibration, evaluation, and SHAP-based interpretation reported in the manuscript.

Input file:
StrokeDataRepositoryV3.xlsx

Primary outputs:
- StrokeData_ML_dataset.xlsx
- Bivariate analysis table
- ECDF and PCA figures
- Final MLP performance metrics
- ROC, calibration, learning curve, and confusion matrix figures
- SHAP beeswarm and SHAP age-threshold figures

The independent hold-out test set is used only for final model evaluation and post hoc SHAP interpretation.

#**Dataset Creation**

In [ ]:
from __future__ import annotations

import argparse
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


# -----------------------------
# Canonical source column names / aliases
# -----------------------------
SEX = "Sex"
LKW = "Final Last Known Well"
SMOKING = "Smoking Status"
ASPIRIN = "Aspirin x=yes"
DRUG_SCREEN = "Lab Evidence of Drug Panel (urine tox screen) Often Ethanol appears too but not looking at this per se"
DIAGNOSIS = "Enc - Primary ICD10 Diagnosis (Billing Code)"
RACE = "Race"
PREV_STROKE = "Evidence of Previous Stroke"
DEPRESSION = "Depression x=yes"
INSURANCE = "Insurance C=Commercial, M=Medicare, A=Medicaid, O=Other"
SERVICE = "Service Number = Underserved, No = Served"
ANTICOAGULANT = "Anticoagulant"
TPA_ADMINISTERED_SOURCE = "tPA (alteplase) administered? May be listed as alteplase in Neurology consult. With or without later bleed."
TPA_BLEED_SOURCE = "Evidence of Bleed following tPA? If No tPA state NA "
TARGET = "Discharge Disposition Summary"

LOS = "Calendar Days in Hospital"
AGE = "Age"
LDL = "LDL (mg/dL)"
SBP = "Systolic Extraction"
GLUCOSE = "Blood Glucose (mg/dL)"
NIHSS = "NIHSS"  # input column is now NIHSS; Final NIHSS remains accepted as an alias
TROPONIN = "Troponin 1 (N;Normal; (less than 0.05 ng/mL), H;High)"

SOURCE_ALIASES = {
    SEX: ["Sex"],
    LKW: ["Final Last Known Well", "Last Known Well"],
    SMOKING: ["Smoking Status"],
    ASPIRIN: ["Aspirin x=yes", "Aspirin Use"],
    DRUG_SCREEN: [
        "Lab Evidence of Drug Panel (urine tox screen) Often Ethanol appears too but not looking at this per se",
        "Drug Screen Positive",
    ],
    DIAGNOSIS: ["Enc - Primary ICD10 Diagnosis (Billing Code)", "Primary Diagnosis"],
    RACE: ["Race"],
    PREV_STROKE: ["Evidence of Previous Stroke", "Previous Stroke"],
    DEPRESSION: ["Depression x=yes", "Depression"],
    INSURANCE: ["Insurance C=Commercial, M=Medicare, A=Medicaid, O=Other", "Insurance Type"],
    SERVICE: ["Service Number = Underserved, No = Served", "Service", "Service (Y/N)"],
    TROPONIN: [
        "Troponin",
        "Troponin 1",
        "Troponin 1 (N;Normal; (less than 0.05 ng/mL), H;High)",
    ],
    ANTICOAGULANT: ["Anticoagulant", "Anticoagulant x=yes", "Anticoagulation x=yes"],
    TPA_ADMINISTERED_SOURCE: [
        "tPA (alteplase) administered? May be listed as alteplase in Neurology consult. With or without later bleed.",
        "tPA administered",
        "tPA administered?",
        "tPA (alteplase) administered",
        "Alteplase administered",
    ],
    TPA_BLEED_SOURCE: [
        "Evidence of Bleed following tPA? If No tPA state NA ",
        "Evidence of Bleed following tPA? If No tPA state NA",
        "Evidence of Bleed following tPA",
        "Bleed following tPA",
    ],
    LOS: ["Calendar Days in Hospital", "LOS (Days)", "Hospital Stay (Days)"],
    AGE: ["Age", "Age (Years)"],
    LDL: ["LDL (mg/dL)"],
    SBP: ["Systolic Extraction", "Systolic BP (mmHg)"],
    GLUCOSE: ["Blood Glucose (mg/dL)"],
    NIHSS: ["NIHSS", "Final NIHSS", "Admission NIHSS"],
    TARGET: ["Discharge Disposition Summary", "Discharge Disposition"],
}

SOURCE_CAT_FEATURES = [
    SEX,
    LKW,
    SMOKING,
    ASPIRIN,
    DRUG_SCREEN,
    DIAGNOSIS,
    RACE,
    PREV_STROKE,
    DEPRESSION,
    INSURANCE,
    SERVICE,
    TROPONIN,
    ANTICOAGULANT,
    TPA_ADMINISTERED_SOURCE,
]

SOURCE_NUM_FEATURES = [LOS, AGE, LDL, SBP, GLUCOSE, NIHSS]

# Professional output column names
COLUMN_RENAME = {
    SEX: "Sex",
    LKW: "Last Known Well",
    SMOKING: "Smoking Status",
    ASPIRIN: "Aspirin Use",
    DRUG_SCREEN: "Drug Screen Positive",
    DIAGNOSIS: "Primary Diagnosis",
    RACE: "Race",
    PREV_STROKE: "Previous Stroke",
    DEPRESSION: "Depression",
    INSURANCE: "Insurance Type",
    SERVICE: "Service",
    TROPONIN: "Troponin",
    ANTICOAGULANT: "Anticoagulant",
    TPA_ADMINISTERED_SOURCE: "tPA administered",
    LOS: "LOS (Days)",
    AGE: "Age (Years)",
    LDL: "LDL (mg/dL)",
    SBP: "Systolic BP (mmHg)",
    GLUCOSE: "Blood Glucose (mg/dL)",
    NIHSS: "Admission NIHSS",
    TARGET: "Discharge Disposition",
}

CAT_FEATURES = [COLUMN_RENAME[c] for c in SOURCE_CAT_FEATURES]
NUM_FEATURES = [COLUMN_RENAME[c] for c in SOURCE_NUM_FEATURES]
TARGET_OUTPUT = COLUMN_RENAME[TARGET]
OUTPUT_COLUMNS = CAT_FEATURES + NUM_FEATURES + [TARGET_OUTPUT]
CAT_SUMMARY_COLUMNS = CAT_FEATURES + [TARGET_OUTPUT]

CATEGORY_ORDER = {
    "Sex": ["Female", "Male"],
    "Last Known Well": ["< 3 hours", "3 to 6 Hours", "6 to 12 hours", "12-24 hours", "> 24 hours"],
    "Smoking Status": ["Never", "Former", "Current"],
    "Aspirin Use": ["No", "Yes"],
    "Drug Screen Positive": ["No", "Yes"],
    "Primary Diagnosis": ["Ischemic", "TIA", "Hemorrhage"],
    "Race": ["White", "Black", "Hispanic", "Other", "Asian"],
    "Previous Stroke": ["No", "Yes"],
    "Depression": ["No", "Yes"],
    "Insurance Type": ["Commercial", "Medicare", "Medicaid", "Other"],
    "Service": ["No", "Yes"],
    "Troponin": ["Normal", "High"],
    "Anticoagulant": ["No", "Yes"],
    "tPA administered": ["No", "Yes - No bleed", "Yes - With bleed"],
    "Discharge Disposition": ["home", "specialized care", "home with help", "expired"],
}

MISSING_TOKENS = {
    "", "na", "n/a", "nan", "none", "null", "missing", "not avail", "not available", "unknown", "unk"
}

TARGET_CLASSES = ["home", "specialized care", "home with help", "expired"]


def norm_text(value: Any) -> str | None:
    """Return lowercase cleaned text, or None for missing-like values."""
    if pd.isna(value):
        return None
    text = str(value).strip()
    text = re.sub(r"\s+", " ", text)
    if text.lower() in MISSING_TOKENS:
        return None
    return text.lower()


def find_column(df: pd.DataFrame, aliases: list[str]) -> str | None:
    """Find a column by exact or case-insensitive alias match after stripping whitespace."""
    exact = {str(c).strip(): c for c in df.columns}
    lower = {str(c).strip().lower(): c for c in df.columns}
    for alias in aliases:
        if alias in exact:
            return exact[alias]
        if alias.lower() in lower:
            return lower[alias.lower()]
    return None


def yes_no_from_x(value: Any) -> str:
    """Columns named x=yes use x/yes/1/true as Yes; blank/missing is treated as No."""
    t = norm_text(value)
    if t in {"x", "yes", "y", "1", "true", "t", "+", "positive", "pos", "present"}:
        return "Yes"
    return "No"


def recode_sex(value: Any) -> str | None:
    t = norm_text(value)
    if t in {"f", "female", "woman"}:
        return "Female"
    if t in {"m", "male", "man"}:
        return "Male"
    return None


def recode_lkw(value: Any) -> str | None:
    t = norm_text(value)
    if t is None:
        return None
    if "less" in t or t in {"< 3 hours", "<3 hours", "under 3 hours"}:
        return "< 3 hours"
    if re.search(r"3\s*(to|-|–)\s*6", t):
        return "3 to 6 Hours"
    if re.search(r"6\s*(to|-|–)\s*12", t):
        return "6 to 12 hours"
    if re.search(r"12\s*(-|to|–)\s*24", t):
        return "12-24 hours"
    if "greater" in t or t in {"> 24 hours", ">24 hours", "over 24 hours"}:
        return "> 24 hours"
    return None


def recode_smoking(value: Any) -> str | None:
    t = norm_text(value)
    if t is None:
        return None
    if t in {"no", "never", "n", "non-smoker", "nonsmoker", "never smoker"}:
        return "Never"
    if t in {"former", "past", "ex", "ex-smoker", "former smoker"}:
        return "Former"
    if t in {"yes", "current", "y", "current smoker", "active"}:
        return "Current"
    return None


def recode_yes_no(value: Any) -> str | None:
    t = norm_text(value)
    if t in {"yes", "y", "1", "true", "t", "x", "+", "positive", "pos", "present"}:
        return "Yes"
    if t in {"no", "n", "0", "false", "f", "-", "negative", "neg", "absent"}:
        return "No"
    return None


def recode_diagnosis(value: Any) -> str | None:
    t = norm_text(value)
    if t is None:
        return None
    if t in {"tia", "transient ischemic attack"} or "transient" in t:
        return "TIA"
    if "isch" in t or "infarct" in t:
        return "Ischemic"
    if "hemorr" in t or "haemorr" in t or "bleed" in t or "intracerebral" in t or t == "ich":
        return "Hemorrhage"
    return None


def recode_race(value: Any) -> str | None:
    t = norm_text(value)
    if t is None:
        return None
    if "white" in t or "caucasian" in t:
        return "White"
    if "black" in t or "african" in t:
        return "Black"
    if "hispanic" in t or "latino" in t or "latina" in t:
        return "Hispanic"
    if "asian" in t:
        return "Asian"
    return "Other"


def recode_insurance(value: Any) -> str | None:
    t = norm_text(value)
    if t is None:
        return None
    if t in {"c", "commercial", "private"} or "commercial" in t:
        return "Commercial"
    if t in {"m", "medicare"} or "medicare" in t:
        return "Medicare"
    if t in {"a", "medicaid"} or "medicaid" in t:
        return "Medicaid"
    if t in {"o", "other", "self pay", "self-pay", "uninsured"}:
        return "Other"
    return "Other"


def recode_service(value: Any) -> str | None:
    """
    Original column meaning:
        service number present = Underserved
        No = Served
    Output:
        Yes = underserved/service-number present
        No = served
    """
    t = norm_text(value)
    if t is None:
        return None
    if t in {"no", "served"}:
        return "No"
    if t in {"yes", "y", "underserved"}:
        return "Yes"
    if re.fullmatch(r"\d+(\.0)?", t):
        return "Yes"
    return None


def recode_troponin(value: Any) -> str | None:
    """
    Original column coding:
        N = Normal / less than 0.05 ng/mL
        H = High
    Output:
        Normal or High
    """
    t = norm_text(value)
    if t is None:
        return None
    if t in {"n", "normal", "negative", "neg", "less than 0.05", "<0.05", "< 0.05"}:
        return "Normal"
    if t in {"h", "high", "positive", "pos", "+", "elevated", "abnormal"}:
        return "High"
    try:
        num = float(re.sub(r"[^0-9.\-]+", "", t))
        if np.isfinite(num):
            return "Normal" if num < 0.05 else "High"
    except Exception:
        pass
    return None


def recode_tpa_administered(tpa_value: Any, bleed_value: Any) -> str | None:
    """
    Build a three-level tPA feature:
        No
        Yes - No bleed
        Yes - With bleed

    The main tPA column is used first. The bleed-after-tPA column is used only
    to validate or resolve values that are recorded as a generic Yes/missing.
    """
    tpa = norm_text(tpa_value)
    bleed = norm_text(bleed_value)

    if tpa in {"no", "n", "0", "false", "f", "not given", "none"}:
        return "No"

    if tpa in {"yes - no bleed", "yes-no bleed", "yes no bleed", "yes without bleed", "yes - without bleed", "no bleed"}:
        return "Yes - No bleed"

    if tpa in {
        "yes - with bleed", "yes-with bleed", "yes with bleed", "yes and bleed", "yes - bleed",
        "with bleed", "bleed"
    }:
        return "Yes - With bleed"

    if tpa in {"yes", "y", "1", "true", "t", "alteplase", "tpa", "t-pa"}:
        if bleed in {"yes", "y", "1", "true", "t", "+", "positive", "pos", "present", "with bleed", "bleed"}:
            return "Yes - With bleed"
        if bleed in {"no", "n", "0", "false", "f", "-", "negative", "neg", "absent", "no bleed"}:
            return "Yes - No bleed"
        return None

    # If the main tPA value is missing but the bleed-after-tPA field is a clear Yes/No,
    # infer that tPA was administered because the bleed field applies after tPA.
    if tpa is None:
        if bleed in {"yes", "y", "1", "true", "t", "+", "positive", "pos", "present", "with bleed", "bleed"}:
            return "Yes - With bleed"
        if bleed in {"no", "n", "0", "false", "f", "-", "negative", "neg", "absent", "no bleed"}:
            return "Yes - No bleed"
        return None

    return None


def recode_disposition(value: Any) -> str | None:
    """
    Robustly map original discharge values into exactly 4 classes:
        home, specialized care, home with help, expired

    Important: this function explicitly preserves already-cleaned target labels
    such as 'specialized care'. Earlier versions missed that exact string, which
    could create blank Discharge Disposition values.
    """
    t = norm_text(value)
    if t is None:
        return None

    # Already-cleaned labels or direct common labels.
    if t in {"home", "home/self care", "home or self care", "routine", "routine discharge", "self care"}:
        return "home"
    if t in {"home with help", "home w help", "home w/ help", "home health", "home with home health", "home care"}:
        return "home with help"
    if t in {"specialized care", "secondary facility", "2ndary facility", "2ndary care", "secondary care"}:
        return "specialized care"
    if t in {"expired", "death", "deceased", "dead", "died"}:
        return "expired"

    # Death/expired first.
    if any(term in t for term in ["expired", "deceased", "death", "dead", "died"]):
        return "expired"

    # Hospice/facility/rehab-type destinations are treated as specialized care.
    if any(term in t for term in [
        "hospice", "2ndary", "secondary", "facility", "rehab", "rehabilitation", "snf",
        "skilled nursing", "nursing", "ltac", "long term acute", "subacute", "specialized"
    ]):
        return "specialized care"

    # Home with services/help.
    if "home" in t and any(term in t for term in [
        "help", "health", "home health", "with", "assist", "assistance", "services", "caregiver"
    ]):
        return "home with help"

    # Other home-like destinations.
    if "home" in t or "self" in t:
        return "home"

    return None


RECODE_FUNCS = {
    SEX: recode_sex,
    LKW: recode_lkw,
    SMOKING: recode_smoking,
    ASPIRIN: yes_no_from_x,
    DRUG_SCREEN: recode_yes_no,
    DIAGNOSIS: recode_diagnosis,
    RACE: recode_race,
    PREV_STROKE: recode_yes_no,
    DEPRESSION: yes_no_from_x,
    INSURANCE: recode_insurance,
    SERVICE: recode_service,
    TROPONIN: recode_troponin,
    ANTICOAGULANT: yes_no_from_x,
    TARGET: recode_disposition,
}


def resolve_columns(df: pd.DataFrame) -> dict[str, str | None]:
    """Resolve all canonical source columns to actual input-file columns."""
    required = SOURCE_CAT_FEATURES + SOURCE_NUM_FEATURES + [TARGET, TPA_BLEED_SOURCE]
    resolved: dict[str, str | None] = {}
    missing_required: list[str] = []

    for canonical in required:
        src = find_column(df, SOURCE_ALIASES[canonical])
        resolved[canonical] = src

        # Anticoagulant is allowed to be absent in older sample files; if absent, output is filled as No.
        # All other source columns must be found.
        if src is None and canonical != ANTICOAGULANT:
            missing_required.append(canonical)

    if missing_required:
        details = []
        for canonical in missing_required:
            aliases = ", ".join(SOURCE_ALIASES[canonical])
            details.append(f"- {canonical}; accepted column name(s): {aliases}")
        raise ValueError(
            "These required columns were not found in the input file:\n" + "\n".join(details)
        )
    return resolved


def validate_no_missing_target(out: pd.DataFrame, df: pd.DataFrame, target_source: str) -> None:
    """Do not allow silent blank target values after recoding."""
    missing_mask = out[TARGET_OUTPUT].isna()
    if not missing_mask.any():
        return

    unmapped = (
        df.loc[missing_mask, target_source]
        .astype("string")
        .fillna("<missing in original file>")
        .value_counts(dropna=False)
        .head(30)
    )
    msg = [
        "Some Discharge Disposition values could not be mapped and would become blank.",
        "The code stopped intentionally so the target column is not silently corrupted.",
        "Unmapped original values and counts:",
    ]
    for value, count in unmapped.items():
        msg.append(f"- {value}: {int(count)}")
    raise ValueError("\n".join(msg))


def validate_no_missing_tpa(out: pd.DataFrame, df: pd.DataFrame, tpa_source: str, bleed_source: str) -> None:
    """Do not allow silent blank tPA administered values after recoding."""
    tpa_output = COLUMN_RENAME[TPA_ADMINISTERED_SOURCE]
    missing_mask = out[tpa_output].isna()
    if not missing_mask.any():
        return

    examples = (
        df.loc[missing_mask, [tpa_source, bleed_source]]
        .astype("string")
        .fillna("<missing in original file>")
        .value_counts(dropna=False)
        .head(30)
    )
    msg = [
        "Some tPA administered values could not be mapped and would become blank.",
        "The code stopped intentionally so the new tPA administered column is not silently corrupted.",
        "Unmapped original tPA / bleed combinations and counts:",
    ]
    for combo, count in examples.items():
        msg.append(f"- {combo}: {int(count)}")
    raise ValueError("\n".join(msg))


def build_ml_dataset(df: pd.DataFrame) -> pd.DataFrame:
    resolved = resolve_columns(df)
    out = pd.DataFrame(index=df.index)

    for source_col in SOURCE_CAT_FEATURES:
        output_col = COLUMN_RENAME[source_col]
        actual_col = resolved[source_col]
        if actual_col is None and source_col == ANTICOAGULANT:
            print("WARNING: Anticoagulant column was not found; Anticoagulant will be filled as 'No'.")
            out[output_col] = "No"
        elif source_col == TPA_ADMINISTERED_SOURCE:
            bleed_col = resolved[TPA_BLEED_SOURCE]
            out[output_col] = df.apply(
                lambda row: recode_tpa_administered(row[actual_col], row[bleed_col]),
                axis=1,
            )
            validate_no_missing_tpa(out, df, actual_col, bleed_col)
        else:
            out[output_col] = df[actual_col].map(RECODE_FUNCS[source_col])

    for source_col in SOURCE_NUM_FEATURES:
        actual_col = resolved[source_col]
        out[COLUMN_RENAME[source_col]] = pd.to_numeric(df[actual_col], errors="coerce")

    target_source = resolved[TARGET]
    out[TARGET_OUTPUT] = df[target_source].map(RECODE_FUNCS[TARGET])
    validate_no_missing_target(out, df, target_source)

    return out[OUTPUT_COLUMNS]


def fmt_count_pct(count: int, denom: int) -> str:
    pct = (count / denom * 100) if denom else 0.0
    return f"{count} ({pct:.1f}%)"


def build_categorical_summary(data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    total_n = len(data)
    for col in CAT_SUMMARY_COLUMNS:
        missing = int(data[col].isna().sum())
        non_missing = total_n - missing
        missing_text = fmt_count_pct(missing, total_n)
        for category in CATEGORY_ORDER[col]:
            count = int((data[col] == category).sum())
            pct = (count / non_missing * 100) if non_missing else 0.0
            rows.append({
                "Variable": col,
                "Category": category,
                "Count": count,
                "Percent of Non-Missing": round(pct, 1),
                "Count (%)": f"{count} ({pct:.1f}%)",
                "Missing Count (%)": missing_text,
            })
    return pd.DataFrame(rows)


def build_numerical_summary(data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    total_n = len(data)
    for col in NUM_FEATURES:
        s = pd.to_numeric(data[col], errors="coerce")
        n = int(s.notna().sum())
        missing = int(s.isna().sum())
        mean = s.mean()
        sd = s.std(ddof=1)
        rows.append({
            "Variable": col,
            "N": n,
            "Mean": round(mean, 2) if pd.notna(mean) else np.nan,
            "SD": round(sd, 2) if pd.notna(sd) else np.nan,
            "Mean ± SD": f"{mean:.2f} ± {sd:.2f}" if pd.notna(mean) and pd.notna(sd) else "",
            "Missing Count (%)": fmt_count_pct(missing, total_n),
        })
    return pd.DataFrame(rows)


def save_outputs(
    selected: pd.DataFrame,
    cat_summary: pd.DataFrame,
    num_summary: pd.DataFrame,
    selected_path: Path,
    summary_path: Path,
) -> None:
    selected.to_excel(selected_path, index=False, sheet_name="ML_Dataset")
    with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:
        cat_summary.to_excel(writer, index=False, sheet_name="Categorical_Summary")
        num_summary.to_excel(writer, index=False, sheet_name="Numerical_Summary")


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description="Create selected stroke ML dataset and summary workbooks.")
    parser.add_argument("--input", "-i", default="StrokeDataRepositoryV3.xlsx", help="Input .xlsx file")
    parser.add_argument("--sheet", default=0, help="Sheet name or index. Default: first sheet")
    parser.add_argument("--selected-output", default="StrokeData_ML_dataset.xlsx")
    parser.add_argument("--summary-output", default="StrokeData_ML_summary.xlsx")
    # Colab/Jupyter adds internal arguments such as -f kernel.json.
    args, _unknown = parser.parse_known_args(argv)

    input_path = Path(args.input)
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    sheet: str | int = int(args.sheet) if str(args.sheet).isdigit() else args.sheet
    df = pd.read_excel(input_path, sheet_name=sheet)

    selected = build_ml_dataset(df)
    cat_summary = build_categorical_summary(selected)
    num_summary = build_numerical_summary(selected)

    save_outputs(selected, cat_summary, num_summary, Path(args.selected_output), Path(args.summary_output))

    print(f"Rows processed: {len(selected):,}")
    print(f"Selected ML dataset: {args.selected_output}")
    print(f"Summary workbook: {args.summary_output}")
    print("\nDischarge Disposition distribution after recoding:")
    print(selected[TARGET_OUTPUT].value_counts(dropna=False).reindex(TARGET_CLASSES).fillna(0).astype(int).to_string())
    print(f"\nMissing Discharge Disposition after recoding: {int(selected[TARGET_OUTPUT].isna().sum())}")


if __name__ == "__main__":
    main()

#**Bivariate Analysis**

In [ ]:
from __future__ import annotations

"""
Bivariate baseline analysis for discharge disposition outcomes.

Output Excel sheets:
  1) Bivariate_Table
     Reported output columns:
       Variable | Category | Count (%) | P-value 1 | P-value 2 | P-value 3
     P-value 1 = home vs specialized care
     P-value 2 = home vs home with help / assistance
     P-value 3 = home vs expired

  2) Pairwise_Test_Details
     Test used, sample sizes, normality p-values, cell counts, etc.

  3) Counts_By_Outcome
     Categorical counts by outcome and numeric summaries by outcome.

  4) Global_Tests
     Optional global tests across all discharge groups.

  5) Notes
     Analysis definitions.

Usage in Colab:
    !python "/content/run_scenario04_bivariate_analysis_with_tpa.py" \
      --input "/content/StrokeData_ML_dataset.xlsx" \
      --sheet 0 \
      --output-excel "/content/scenario04_bivariate_baseline_table.xlsx"
"""

import argparse
import os
import re
import sys
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from scipy.stats import chi2_contingency, fisher_exact, f_oneway, kruskal, shapiro
except Exception as exc:  # pragma: no cover
    raise ImportError(
        "This script requires scipy. In Colab run: !pip install scipy openpyxl"
    ) from exc

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------------------------------------------------------
# Feature definitions: WITH Anticoagulant and tPA administered
# -----------------------------------------------------------------------------

TARGET_CLASSES = ["home", "specialized care", "home with help", "expired"]
REFERENCE_CLASS = "home"
PAIRWISE_COMPARISONS = [
    ("P-value 1", "home", "specialized care"),
    ("P-value 2", "home", "home with help"),
    ("P-value 3", "home", "expired"),
]

FEATURE_ALIASES = {
    "Sex": ["Sex"],
    "Last Known Well": ["Last Known Well", "Final Last Known Well"],
    "Smoking Status": ["Smoking Status"],
    "Aspirin Use": ["Aspirin Use", "Aspirin x=yes"],
    "Anticoagulant": ["Anticoagulant", "Anticoagulant x=yes", "Anticoagulant x = yes"],
    "tPA administered": [
        "tPA administered",
        "tPA administered?",
        "tPA (alteplase) administered",
        "Alteplase administered",
        "tPA (alteplase) administered? May be listed as alteplase in Neurology consult. With or without later bleed.",
    ],
    "Drug Screen Positive": [
        "Drug Screen Positive",
        "Lab Evidence of Drug Panel (urine tox screen) Often Ethanol appears too but not looking at this per se",
    ],
    "Primary Diagnosis": ["Primary Diagnosis", "Enc - Primary ICD10 Diagnosis (Billing Code)"],
    "Race": ["Race"],
    "Previous Stroke": ["Previous Stroke", "Evidence of Previous Stroke"],
    "Depression": ["Depression", "Depression x=yes"],
    "Insurance Type": ["Insurance Type", "Insurance C=Commercial, M=Medicare, A=Medicaid, O=Other"],
    "Service": ["Service", "Service (Y/N)", "Service Number = Underserved, No = Served"],
    "Troponin": [
        "Troponin",
        "Troponin 1",
        "Troponin 1 (N;Normal; (less than 0.05 ng/mL), H;High)",
    ],
    "LOS (Days)": ["LOS (Days)", "Hospital Stay (Days)", "Calendar Days in Hospital"],
    "Age (Years)": ["Age (Years)", "Age"],
    "LDL (mg/dL)": ["LDL (mg/dL)"],
    "Systolic BP (mmHg)": ["Systolic BP (mmHg)", "Systolic Extraction"],
    "Blood Glucose (mg/dL)": ["Blood Glucose (mg/dL)"],
    "Admission NIHSS": ["Admission NIHSS", "Final NIHSS", "NIHSS"],
}

TPA_BLEED_ALIASES = [
    "Evidence of Bleed following tPA? If No tPA state NA ",
    "Evidence of Bleed following tPA? If No tPA state NA",
    "Evidence of Bleed following tPA",
    "Bleed following tPA",
]

TARGET_ALIASES = ["Discharge Disposition", "Discharge Disposition Summary"]

NUMERIC_FEATURES = [
    "LOS (Days)",
    "Age (Years)",
    "LDL (mg/dL)",
    "Systolic BP (mmHg)",
    "Blood Glucose (mg/dL)",
    "Admission NIHSS",
]

ORDINAL_FEATURES = [
    "Last Known Well",
    "Smoking Status",
    "Troponin",
]

BINARY_FEATURES = [
    "Sex",
    "Aspirin Use",
    "Anticoagulant",
    "Drug Screen Positive",
    "Previous Stroke",
    "Depression",
    "Service",
]

NOMINAL_FEATURES = ["Primary Diagnosis", "Race", "Insurance Type", "tPA administered"]
CATEGORICAL_FEATURES = ORDINAL_FEATURES + BINARY_FEATURES + NOMINAL_FEATURES
ALL_FEATURES = list(FEATURE_ALIASES.keys())

ORDINAL_CATEGORIES = {
    "Last Known Well": ["< 3 hours", "3 to 6 hours", "6 to 12 hours", "12-24 hours", "> 24 hours"],
    "Smoking Status": ["Never", "Former", "Current"],
    "Troponin": ["Normal", "High"],
}

BINARY_CATEGORIES = {
    "Sex": ["Female", "Male"],
    "Aspirin Use": ["No", "Yes"],
    "Anticoagulant": ["No", "Yes"],
    "Drug Screen Positive": ["No", "Yes"],
    "Previous Stroke": ["No", "Yes"],
    "Depression": ["No", "Yes"],
    "Service": ["No", "Yes"],
}

KNOWN_NOMINAL_CATEGORIES = {
    "Primary Diagnosis": ["Ischemic", "TIA", "Hemorrhage"],
    "Race": ["White", "Black", "Hispanic", "Asian", "Other"],
    "Insurance Type": ["Commercial", "Medicare", "Medicaid", "Other"],
    "tPA administered": ["No", "Yes - No bleed", "Yes - With bleed"],
}

KNOWN_CATEGORIES: Dict[str, List[str]] = {}
KNOWN_CATEGORIES.update(ORDINAL_CATEGORIES)
KNOWN_CATEGORIES.update(BINARY_CATEGORIES)
KNOWN_CATEGORIES.update(KNOWN_NOMINAL_CATEGORIES)

MISSING_STRINGS = {
    "", " ", "nan", "none", "null", "na", "n/a", "not avail", "not available", "unknown", "unk"
}

# -----------------------------------------------------------------------------
# Cleaning and standardization
# -----------------------------------------------------------------------------

def normalize_text(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    if s.lower() in MISSING_STRINGS:
        return np.nan
    return s


def find_column(df: pd.DataFrame, aliases: List[str]) -> Optional[str]:
    exact = {str(c).strip(): c for c in df.columns}
    lower = {str(c).strip().lower(): c for c in df.columns}
    for alias in aliases:
        if alias in exact:
            return exact[alias]
        if alias.lower() in lower:
            return lower[alias.lower()]
    return None


def normalize_yes_no(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    yes_values = {"yes", "y", "true", "1", "x", "positive", "pos", "+", "present", "underserved"}
    no_values = {"no", "n", "false", "0", "negative", "neg", "-", "absent", "served"}
    if sl in yes_values:
        return "Yes"
    if sl in no_values:
        return "No"
    return np.nan


def normalize_sex(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"f", "female", "woman"}:
        return "Female"
    if sl in {"m", "male", "man"}:
        return "Male"
    return np.nan


def normalize_lkw(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).lower().replace("hrs", "hours").replace("hour", "hours").strip()
    sl = re.sub(r"\s+", " ", sl)
    if "<" in sl and "3" in sl:
        return "< 3 hours"
    if "less" in sl and "3" in sl:
        return "< 3 hours"
    if ">" in sl and "24" in sl:
        return "> 24 hours"
    if "greater" in sl and "24" in sl:
        return "> 24 hours"
    if ("3" in sl and "6" in sl) or sl in {"3-6", "3 to 6"}:
        return "3 to 6 hours"
    if ("6" in sl and "12" in sl) or sl in {"6-12", "6 to 12"}:
        return "6 to 12 hours"
    if ("12" in sl and "24" in sl) or sl in {"12-24", "12 to 24"}:
        return "12-24 hours"
    return np.nan


def normalize_smoking(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"never", "no", "none", "non-smoker", "nonsmoker", "never smoker"}:
        return "Never"
    if sl in {"former", "previous", "past", "ex-smoker", "ex smoker", "former smoker"}:
        return "Former"
    if sl in {"current", "yes", "active", "smoker", "current smoker"}:
        return "Current"
    return np.nan


def normalize_diagnosis(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).lower()
    if "tia" in sl or "transient" in sl:
        return "TIA"
    if "hemorr" in sl or "haemorr" in sl or "ich" in sl or "sah" in sl or "bleed" in sl:
        return "Hemorrhage"
    if "ischem" in sl or "ischaem" in sl or "infarct" in sl or sl.startswith("i"):
        return "Ischemic"
    return np.nan


def normalize_race(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).lower()
    if "white" in sl or "cauc" in sl:
        return "White"
    if "black" in sl or "african" in sl:
        return "Black"
    if "hispanic" in sl or "latino" in sl or "latina" in sl:
        return "Hispanic"
    if "asian" in sl:
        return "Asian"
    return "Other"


def normalize_insurance(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"c", "commercial", "private"} or "commercial" in sl:
        return "Commercial"
    if sl in {"m", "medicare"} or "medicare" in sl:
        return "Medicare"
    if sl in {"a", "medicaid"} or "medicaid" in sl:
        return "Medicaid"
    if sl in {"o", "other", "self pay", "self-pay", "uninsured"}:
        return "Other"
    return "Other"


def normalize_troponin(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"n", "normal", "negative", "neg", "0"}:
        return "Normal"
    if sl in {"h", "high", "positive", "pos", "1", "elevated", "abnormal"}:
        return "High"
    cleaned = sl.replace("ng/ml", "")
    if "less" in cleaned or "<" in cleaned:
        nums = re.findall(r"[-+]?\d*\.?\d+", cleaned)
        if nums:
            try:
                if float(nums[0]) <= 0.05:
                    return "Normal"
            except Exception:
                pass
    nums = re.findall(r"[-+]?\d*\.?\d+", cleaned)
    if nums:
        try:
            val = float(nums[0])
            return "High" if val >= 0.05 else "Normal"
        except Exception:
            return np.nan
    return np.nan


def normalize_tpa_administered(tpa_value, bleed_value=None):
    """
    Normalize tPA into exactly three classes:
        No
        Yes - No bleed
        Yes - With bleed

    If the input file already contains a cleaned `tPA administered` column, this
    simply preserves/standardizes it. If the file still has the original raw tPA
    column, the bleed-after-tPA column is used only to resolve generic Yes/missing
    values when available.
    """
    tpa = normalize_text(tpa_value)
    bleed = normalize_text(bleed_value) if bleed_value is not None else np.nan

    tpa_l = str(tpa).strip().lower() if not pd.isna(tpa) else ""
    bleed_l = str(bleed).strip().lower() if not pd.isna(bleed) else ""

    if tpa_l in {"no", "n", "0", "false", "f", "not given", "none"}:
        return "No"

    if tpa_l in {"yes - no bleed", "yes-no bleed", "yes no bleed", "yes without bleed", "yes - without bleed", "no bleed"}:
        return "Yes - No bleed"

    if tpa_l in {
        "yes - with bleed", "yes-with bleed", "yes with bleed", "yes and bleed", "yes - bleed",
        "with bleed", "bleed"
    }:
        return "Yes - With bleed"

    if tpa_l in {"yes", "y", "1", "true", "t", "alteplase", "tpa", "t-pa"}:
        if bleed_l in {"yes", "y", "1", "true", "t", "+", "positive", "pos", "present", "with bleed", "bleed"}:
            return "Yes - With bleed"
        if bleed_l in {"no", "n", "0", "false", "f", "-", "negative", "neg", "absent", "no bleed"}:
            return "Yes - No bleed"
        return np.nan

    if tpa_l == "":
        if bleed_l in {"yes", "y", "1", "true", "t", "+", "positive", "pos", "present", "with bleed", "bleed"}:
            return "Yes - With bleed"
        if bleed_l in {"no", "n", "0", "false", "f", "-", "negative", "neg", "absent", "no bleed"}:
            return "Yes - No bleed"
        return np.nan

    return np.nan


def normalize_target(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    sl = re.sub(r"\s+", " ", sl)
    if sl == "home":
        return "home"
    if sl in {"home with help", "home w help", "home w/ help", "home health", "home with home health", "home with assistance"}:
        return "home with help"
    if sl in {"specialized care", "snf", "rehab", "acute rehab", "skilled nursing", "ltac", "subacute rehab"}:
        return "specialized care"
    if "rehab" in sl or "snf" in sl or "nursing" in sl or "facility" in sl or "hospice" in sl:
        return "specialized care"
    if sl in {"expired", "death", "deceased", "dead", "died"}:
        return "expired"
    return np.nan


def standardize_dataframe(df_raw: pd.DataFrame) -> Tuple[pd.DataFrame, str]:
    data = pd.DataFrame(index=df_raw.index)

    missing_cols = []
    for canonical, aliases in FEATURE_ALIASES.items():
        src = find_column(df_raw, aliases)
        if src is None:
            missing_cols.append(canonical)
        elif canonical == "tPA administered":
            bleed_src = find_column(df_raw, TPA_BLEED_ALIASES)
            if bleed_src is None:
                data[canonical] = df_raw[src].map(lambda x: normalize_tpa_administered(x, None))
            else:
                data[canonical] = [
                    normalize_tpa_administered(tpa_value, bleed_value)
                    for tpa_value, bleed_value in zip(df_raw[src], df_raw[bleed_src])
                ]
        else:
            data[canonical] = df_raw[src]

    target_src = find_column(df_raw, TARGET_ALIASES)
    if target_src is None:
        raise ValueError(f"Target column was not found. Expected one of: {TARGET_ALIASES}")
    data["Discharge Disposition"] = df_raw[target_src]

    if missing_cols:
        raise ValueError(
            "The following feature columns were not found: " + ", ".join(missing_cols)
        )

    for col in NUMERIC_FEATURES:
        data[col] = pd.to_numeric(data[col].replace(list(MISSING_STRINGS), np.nan), errors="coerce")

    data["Sex"] = data["Sex"].map(normalize_sex)
    data["Last Known Well"] = data["Last Known Well"].map(normalize_lkw)
    data["Smoking Status"] = data["Smoking Status"].map(normalize_smoking)
    for col in [c for c in BINARY_FEATURES if c != "Sex"]:
        data[col] = data[col].map(normalize_yes_no)
    data["Primary Diagnosis"] = data["Primary Diagnosis"].map(normalize_diagnosis)
    data["Race"] = data["Race"].map(normalize_race)
    data["Insurance Type"] = data["Insurance Type"].map(normalize_insurance)
    data["Troponin"] = data["Troponin"].map(normalize_troponin)
    data["Discharge Disposition"] = data["Discharge Disposition"].map(normalize_target)

    before = len(data)
    data = data[data["Discharge Disposition"].isin(TARGET_CLASSES)].copy()
    dropped = before - len(data)
    if dropped > 0:
        print(f"Dropped {dropped} rows because target was missing or outside the 4 requested classes.")

    return data, "Discharge Disposition"

# -----------------------------------------------------------------------------
# Statistical helpers
# -----------------------------------------------------------------------------

def fmt_pvalue(p) -> str:
    if p is None or not np.isfinite(p):
        return "NA"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def fmt_count_pct(n: int, denom: int) -> str:
    if denom <= 0:
        return f"{n} (NA)"
    return f"{n} ({100.0 * n / denom:.1f}%)"


def fmt_numeric_summary(values: pd.Series, normal: Optional[bool] = None) -> str:
    x = pd.to_numeric(values, errors="coerce").dropna().astype(float)
    if len(x) == 0:
        return "NA"
    if normal is None:
        normal = shapiro_is_normal(x)[0]
    if normal:
        return f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"
    q1, med, q3 = np.percentile(x, [25, 50, 75])
    return f"{med:.2f} [{q1:.2f}, {q3:.2f}]"


def shapiro_is_normal(values: pd.Series, alpha: float = 0.05, random_state: int = 42) -> Tuple[bool, float, int]:
    """Return (is_normal, p_value, n_used). Uses Shapiro-Wilk; samples to 5000 if larger."""
    x = pd.to_numeric(values, errors="coerce").dropna().astype(float).to_numpy()
    x = x[np.isfinite(x)]
    if len(x) < 3:
        return False, np.nan, int(len(x))
    if len(np.unique(x)) < 3:
        return False, np.nan, int(len(x))
    n_used = len(x)
    if len(x) > 5000:
        rng = np.random.default_rng(random_state)
        x = rng.choice(x, size=5000, replace=False)
        n_used = 5000
    try:
        _, p = shapiro(x)
        return bool(p >= alpha), float(p), int(n_used)
    except Exception:
        return False, np.nan, int(n_used)


def contingency_for_feature(df: pd.DataFrame, feature: str, outcome_col: str, classes: List[str]) -> pd.DataFrame:
    sub = df[[feature, outcome_col]].dropna().copy()
    sub = sub[sub[outcome_col].isin(classes)]
    if sub.empty:
        return pd.DataFrame()
    tab = pd.crosstab(sub[outcome_col], sub[feature])
    for cls in classes:
        if cls not in tab.index:
            tab.loc[cls] = 0
    # Preserve known category order where available, then append unexpected observed categories.
    ordered_cols = []
    for cat in KNOWN_CATEGORIES.get(feature, []):
        if cat in tab.columns:
            ordered_cols.append(cat)
    for cat in tab.columns:
        if cat not in ordered_cols:
            ordered_cols.append(cat)
    tab = tab.reindex(index=classes, columns=ordered_cols, fill_value=0)
    tab = tab.loc[:, tab.sum(axis=0) > 0]
    return tab.astype(int)


def monte_carlo_fixed_margin_chi2_p(
    observed: np.ndarray,
    n_iter: int = 20000,
    random_state: int = 42,
) -> Tuple[float, float]:
    """
    Monte-Carlo exact-style p-value for an RxC table with fixed margins.

    This approximates a Fisher-Freeman-Halton style exact test by sampling random
    tables with the same row and column margins and comparing Pearson chi-square
    statistics. It is used when Fisher's exact test is requested but the table is
    larger than 2x2.
    """
    observed = np.asarray(observed, dtype=int)
    if observed.ndim != 2 or observed.shape[0] < 2 or observed.shape[1] < 2:
        return np.nan, np.nan
    try:
        obs_stat = float(chi2_contingency(observed, correction=False)[0])
    except Exception:
        return np.nan, np.nan

    row_sums = observed.sum(axis=1).astype(int)
    col_sums = observed.sum(axis=0).astype(int)
    total = int(observed.sum())
    if total == 0:
        return np.nan, obs_stat

    rng = np.random.default_rng(random_state)
    ge = 0
    completed = 0

    for _ in range(int(n_iter)):
        remaining_cols = col_sums.copy()
        sim = np.zeros_like(observed, dtype=int)
        valid = True
        for r in range(observed.shape[0] - 1):
            row_n = int(row_sums[r])
            if row_n == 0:
                continue
            population = np.repeat(np.arange(observed.shape[1]), remaining_cols)
            if len(population) < row_n:
                valid = False
                break
            draw = rng.choice(population, size=row_n, replace=False)
            counts = np.bincount(draw, minlength=observed.shape[1])
            sim[r, :] = counts
            remaining_cols -= counts
            if np.any(remaining_cols < 0):
                valid = False
                break
        if not valid:
            continue
        sim[-1, :] = remaining_cols
        try:
            sim_stat = float(chi2_contingency(sim, correction=False)[0])
        except Exception:
            continue
        completed += 1
        if sim_stat >= obs_stat - 1e-12:
            ge += 1

    if completed == 0:
        return np.nan, obs_stat
    p = (ge + 1.0) / (completed + 1.0)
    return float(p), obs_stat


def categorical_pvalue(
    df: pd.DataFrame,
    feature: str,
    outcome_col: str,
    class_a: str,
    class_b: str,
    fisher_mc_iter: int,
    random_state: int,
) -> Dict[str, object]:
    tab = contingency_for_feature(df, feature, outcome_col, [class_a, class_b])
    result = {
        "P-value": np.nan,
        "P-value formatted": "NA",
        "Test": "Insufficient data",
        "Minimum observed cell": np.nan,
        "Minimum expected cell": np.nan,
        "Table shape": "NA",
        "Chi-square statistic": np.nan,
        "N analyzed": int(tab.values.sum()) if not tab.empty else 0,
    }
    if tab.empty or tab.shape[0] < 2 or tab.shape[1] < 2 or tab.values.sum() == 0:
        return result

    observed = tab.values.astype(int)
    result["Minimum observed cell"] = int(observed.min())
    result["Table shape"] = f"{observed.shape[0]}x{observed.shape[1]}"

    try:
        chi2_stat, chi2_p, _, expected = chi2_contingency(observed, correction=False)
        min_expected = float(np.min(expected))
        result["Minimum expected cell"] = min_expected
        result["Chi-square statistic"] = float(chi2_stat)
    except Exception:
        chi2_p = np.nan
        min_expected = np.nan
        expected = None

    low_frequency = bool(np.any(observed < 5)) or (expected is not None and bool(np.any(expected < 5)))

    if low_frequency:
        if observed.shape == (2, 2):
            try:
                _, p = fisher_exact(observed)
                result["P-value"] = float(p)
                result["P-value formatted"] = fmt_pvalue(float(p))
                result["Test"] = "Fisher exact test"
            except Exception:
                result["P-value"] = np.nan
                result["P-value formatted"] = "NA"
                result["Test"] = "Fisher exact test failed"
        else:
            p, stat = monte_carlo_fixed_margin_chi2_p(
                observed,
                n_iter=fisher_mc_iter,
                random_state=random_state,
            )
            result["P-value"] = float(p) if np.isfinite(p) else np.nan
            result["P-value formatted"] = fmt_pvalue(p)
            result["Test"] = f"Fisher-Freeman-Halton Monte Carlo approximation ({fisher_mc_iter:,} draws)"
            result["Chi-square statistic"] = stat
    else:
        result["P-value"] = float(chi2_p) if np.isfinite(chi2_p) else np.nan
        result["P-value formatted"] = fmt_pvalue(chi2_p)
        result["Test"] = "Pearson chi-square test"

    return result


def numeric_pairwise_pvalue(
    df: pd.DataFrame,
    feature: str,
    outcome_col: str,
    class_a: str,
    class_b: str,
    alpha: float,
    random_state: int,
) -> Dict[str, object]:
    sub = df[[feature, outcome_col]].dropna().copy()
    sub = sub[sub[outcome_col].isin([class_a, class_b])]
    x_a = pd.to_numeric(sub.loc[sub[outcome_col] == class_a, feature], errors="coerce").dropna().astype(float)
    x_b = pd.to_numeric(sub.loc[sub[outcome_col] == class_b, feature], errors="coerce").dropna().astype(float)

    normal_a, shapiro_a, n_shapiro_a = shapiro_is_normal(x_a, alpha=alpha, random_state=random_state)
    normal_b, shapiro_b, n_shapiro_b = shapiro_is_normal(x_b, alpha=alpha, random_state=random_state + 1)
    both_normal = normal_a and normal_b

    result = {
        "P-value": np.nan,
        "P-value formatted": "NA",
        "Test": "Insufficient data",
        "Reference N": int(len(x_a)),
        "Comparison N": int(len(x_b)),
        "Shapiro P reference": shapiro_a,
        "Shapiro P comparison": shapiro_b,
        "Shapiro N reference": n_shapiro_a,
        "Shapiro N comparison": n_shapiro_b,
        "Normal reference": normal_a,
        "Normal comparison": normal_b,
        "Reference summary": fmt_numeric_summary(x_a, normal=normal_a),
        "Comparison summary": fmt_numeric_summary(x_b, normal=normal_b),
    }

    if len(x_a) < 2 or len(x_b) < 2:
        return result

    try:
        if both_normal:
            stat, p = f_oneway(x_a, x_b)  # two-group one-way ANOVA
            result["Test"] = "One-way ANOVA"
        else:
            stat, p = kruskal(x_a, x_b)
            result["Test"] = "Kruskal-Wallis test"
        result["P-value"] = float(p)
        result["P-value formatted"] = fmt_pvalue(float(p))
        result["Statistic"] = float(stat)
    except Exception:
        result["Test"] = "Test failed"
    return result


def global_pvalue(
    df: pd.DataFrame,
    feature: str,
    outcome_col: str,
    feature_type: str,
    alpha: float,
    fisher_mc_iter: int,
    random_state: int,
) -> Dict[str, object]:
    if feature_type == "categorical":
        tab = contingency_for_feature(df, feature, outcome_col, TARGET_CLASSES)
        result = {
            "Variable": feature,
            "Feature Type": feature_type,
            "Test": "Insufficient data",
            "P-value": np.nan,
            "P-value formatted": "NA",
            "N analyzed": int(tab.values.sum()) if not tab.empty else 0,
            "Minimum observed cell": np.nan,
            "Minimum expected cell": np.nan,
            "Table shape": "NA",
        }
        if tab.empty or tab.shape[0] < 2 or tab.shape[1] < 2:
            return result
        observed = tab.values.astype(int)
        result["Minimum observed cell"] = int(observed.min())
        result["Table shape"] = f"{observed.shape[0]}x{observed.shape[1]}"
        try:
            chi2_stat, chi2_p, _, expected = chi2_contingency(observed, correction=False)
            result["Minimum expected cell"] = float(np.min(expected))
            low_frequency = bool(np.any(observed < 5)) or bool(np.any(expected < 5))
        except Exception:
            chi2_p = np.nan
            low_frequency = True
        if low_frequency:
            p, _ = monte_carlo_fixed_margin_chi2_p(observed, n_iter=fisher_mc_iter, random_state=random_state)
            result["P-value"] = float(p) if np.isfinite(p) else np.nan
            result["P-value formatted"] = fmt_pvalue(p)
            result["Test"] = f"Fisher-Freeman-Halton Monte Carlo approximation ({fisher_mc_iter:,} draws)"
        else:
            result["P-value"] = float(chi2_p) if np.isfinite(chi2_p) else np.nan
            result["P-value formatted"] = fmt_pvalue(chi2_p)
            result["Test"] = "Pearson chi-square test"
        return result

    # Numeric global test across all available outcome classes.
    sub = df[[feature, outcome_col]].dropna().copy()
    groups = []
    normality_parts = []
    all_normal = True
    for cls in TARGET_CLASSES:
        x = pd.to_numeric(sub.loc[sub[outcome_col] == cls, feature], errors="coerce").dropna().astype(float)
        if len(x) > 0:
            groups.append((cls, x))
            normal, p_norm, n_norm = shapiro_is_normal(x, alpha=alpha, random_state=random_state)
            all_normal = all_normal and normal
            normality_parts.append(f"{cls}: p={fmt_pvalue(p_norm)}, n={n_norm}, normal={normal}")
    result = {
        "Variable": feature,
        "Feature Type": feature_type,
        "Test": "Insufficient data",
        "P-value": np.nan,
        "P-value formatted": "NA",
        "N analyzed": int(sum(len(g[1]) for g in groups)),
        "Normality by group": "; ".join(normality_parts),
    }
    groups_for_test = [g[1] for g in groups if len(g[1]) >= 2]
    if len(groups_for_test) < 2:
        return result
    try:
        if all_normal:
            stat, p = f_oneway(*groups_for_test)
            result["Test"] = "One-way ANOVA"
        else:
            stat, p = kruskal(*groups_for_test)
            result["Test"] = "Kruskal-Wallis test"
        result["P-value"] = float(p)
        result["P-value formatted"] = fmt_pvalue(float(p))
        result["Statistic"] = float(stat)
    except Exception:
        result["Test"] = "Test failed"
    return result

# -----------------------------------------------------------------------------
# Table builders
# -----------------------------------------------------------------------------

def build_bivariate_tables(
    data: pd.DataFrame,
    target_col: str,
    alpha: float,
    fisher_mc_iter: int,
    random_state: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    main_rows = []
    pairwise_detail_rows = []
    count_detail_rows = []
    global_rows = []
    missing_rows = []

    # Precompute pairwise p-values for every feature.
    pairwise_results: Dict[str, Dict[str, Dict[str, object]]] = {}

    for feature in ALL_FEATURES:
        if feature in CATEGORICAL_FEATURES:
            feature_type = "categorical"
            pairwise_results[feature] = {}
            for p_col, ref, comp in PAIRWISE_COMPARISONS:
                res = categorical_pvalue(
                    data, feature, target_col, ref, comp,
                    fisher_mc_iter=fisher_mc_iter,
                    random_state=random_state + hash((feature, p_col)) % 100000,
                )
                pairwise_results[feature][p_col] = res
                pairwise_detail_rows.append({
                    "Variable": feature,
                    "Feature Type": feature_type,
                    "Comparison": f"{ref} vs {comp}",
                    "P-value column": p_col,
                    "Test": res.get("Test"),
                    "P-value": res.get("P-value"),
                    "P-value formatted": res.get("P-value formatted"),
                    "N analyzed": res.get("N analyzed"),
                    "Minimum observed cell": res.get("Minimum observed cell"),
                    "Minimum expected cell": res.get("Minimum expected cell"),
                    "Table shape": res.get("Table shape"),
                    "Reference group": ref,
                    "Comparison group": comp,
                })
            global_rows.append(global_pvalue(
                data, feature, target_col, feature_type,
                alpha=alpha,
                fisher_mc_iter=fisher_mc_iter,
                random_state=random_state + hash(feature) % 100000,
            ))

            nonmissing = data[feature].dropna()
            denom = int(len(nonmissing))
            known_order = KNOWN_CATEGORIES.get(feature, [])
            observed_order = [v for v in nonmissing.astype(str).unique().tolist() if v not in known_order]
            categories = known_order + sorted(observed_order)
            categories = [cat for cat in categories if (nonmissing == cat).sum() > 0]
            if len(categories) == 0:
                categories = ["No non-missing data"]

            for cat in categories:
                n = int((data[feature] == cat).sum()) if cat != "No non-missing data" else 0
                row = {
                    "Variable": feature,
                    "Category": cat,
                    "Count (%)": fmt_count_pct(n, denom),
                }
                for p_col, _, _ in PAIRWISE_COMPARISONS:
                    row[p_col] = pairwise_results[feature][p_col].get("P-value formatted", "NA")
                main_rows.append(row)

                detail = {"Variable": feature, "Category": cat, "Total": n, "Total % of non-missing": (100.0 * n / denom if denom else np.nan)}
                for cls in TARGET_CLASSES:
                    cls_mask = data[target_col] == cls
                    cls_denom = int(data.loc[cls_mask, feature].notna().sum())
                    cls_n = int(((data[feature] == cat) & cls_mask).sum()) if cat != "No non-missing data" else 0
                    detail[f"{cls} N"] = cls_n
                    detail[f"{cls} %"] = (100.0 * cls_n / cls_denom if cls_denom else np.nan)
                count_detail_rows.append(detail)

            n_missing = int(data[feature].isna().sum())
            missing_rows.append({
                "Variable": feature,
                "Feature Type": feature_type,
                "Non-missing N": denom,
                "Missing N": n_missing,
                "Missing %": 100.0 * n_missing / len(data) if len(data) else np.nan,
            })

        elif feature in NUMERIC_FEATURES:
            feature_type = "numeric"
            pairwise_results[feature] = {}
            for p_col, ref, comp in PAIRWISE_COMPARISONS:
                res = numeric_pairwise_pvalue(
                    data, feature, target_col, ref, comp,
                    alpha=alpha,
                    random_state=random_state + hash((feature, p_col)) % 100000,
                )
                pairwise_results[feature][p_col] = res
                pairwise_detail_rows.append({
                    "Variable": feature,
                    "Feature Type": feature_type,
                    "Comparison": f"{ref} vs {comp}",
                    "P-value column": p_col,
                    "Test": res.get("Test"),
                    "P-value": res.get("P-value"),
                    "P-value formatted": res.get("P-value formatted"),
                    "Reference N": res.get("Reference N"),
                    "Comparison N": res.get("Comparison N"),
                    "Reference summary": res.get("Reference summary"),
                    "Comparison summary": res.get("Comparison summary"),
                    "Shapiro P reference": res.get("Shapiro P reference"),
                    "Shapiro P comparison": res.get("Shapiro P comparison"),
                    "Normal reference": res.get("Normal reference"),
                    "Normal comparison": res.get("Normal comparison"),
                    "Reference group": ref,
                    "Comparison group": comp,
                })
            global_rows.append(global_pvalue(
                data, feature, target_col, feature_type,
                alpha=alpha,
                fisher_mc_iter=fisher_mc_iter,
                random_state=random_state + hash(feature) % 100000,
            ))

            x = pd.to_numeric(data[feature], errors="coerce")
            normal_overall, shapiro_p, shapiro_n = shapiro_is_normal(x, alpha=alpha, random_state=random_state)
            summary = fmt_numeric_summary(x, normal=normal_overall)
            row = {
                "Variable": feature,
                "Category": "Continuous",
                "Count (%)": summary,
            }
            for p_col, _, _ in PAIRWISE_COMPARISONS:
                row[p_col] = pairwise_results[feature][p_col].get("P-value formatted", "NA")
            main_rows.append(row)

            detail = {
                "Variable": feature,
                "Category": "Continuous",
                "Total": int(x.notna().sum()),
                "Total summary": summary,
                "Overall Shapiro-Wilk P": shapiro_p,
                "Overall Shapiro-Wilk N used": shapiro_n,
                "Overall distribution used for display": "Mean ± SD" if normal_overall else "Median [IQR]",
            }
            for cls in TARGET_CLASSES:
                vals = pd.to_numeric(data.loc[data[target_col] == cls, feature], errors="coerce")
                normal_cls, _, _ = shapiro_is_normal(vals, alpha=alpha, random_state=random_state)
                detail[f"{cls} N"] = int(vals.notna().sum())
                detail[f"{cls} summary"] = fmt_numeric_summary(vals, normal=normal_cls)
            count_detail_rows.append(detail)

            n_missing = int(x.isna().sum())
            missing_rows.append({
                "Variable": feature,
                "Feature Type": feature_type,
                "Non-missing N": int(x.notna().sum()),
                "Missing N": n_missing,
                "Missing %": 100.0 * n_missing / len(data) if len(data) else np.nan,
                "Overall Shapiro-Wilk P": shapiro_p,
                "Overall Shapiro-Wilk N used": shapiro_n,
            })

    main_df = pd.DataFrame(main_rows, columns=["Variable", "Category", "Count (%)", "P-value 1", "P-value 2", "P-value 3"])
    pairwise_df = pd.DataFrame(pairwise_detail_rows)
    counts_df = pd.DataFrame(count_detail_rows)
    global_df = pd.DataFrame(global_rows)
    missing_df = pd.DataFrame(missing_rows)

    return main_df, pairwise_df, counts_df, global_df, missing_df


def build_notes_df(input_path: str, n_rows: int) -> pd.DataFrame:
    notes = [
        ("Input file", input_path),
        ("Rows analyzed", n_rows),
        ("Scenario", "Scenario 04: with Anticoagulant and tPA administered; no MLP tuning or model fitting is performed here."),
        ("Reference category", "home"),
        ("P-value 1", "home vs specialized care"),
        ("P-value 2", "home vs home with help / assistance"),
        ("P-value 3", "home vs expired"),
        ("tPA administered", "Categorical three-level feature: No, Yes - No bleed, Yes - With bleed."),
        ("Categorical tests", "Pearson chi-square unless any observed or expected cell <5. For 2x2 low-frequency tables, Fisher exact test is used. For larger low-frequency tables, a Fisher-Freeman-Halton style Monte Carlo fixed-margin approximation is used."),
        ("Numeric tests", "Shapiro-Wilk tests are run within groups. If both compared groups are normal, one-way ANOVA is used for the pairwise comparison; otherwise Kruskal-Wallis is used."),
        ("Numeric display", "Normal numeric features are shown as mean ± SD; non-normal numeric features are shown as median [IQR] in the Count (%) column."),
        ("Missing data", "P-values are computed using complete cases for the feature and outcome being tested."),
    ]
    return pd.DataFrame(notes, columns=["Item", "Definition"])


def write_excel(
    output_excel: str,
    main_df: pd.DataFrame,
    pairwise_df: pd.DataFrame,
    counts_df: pd.DataFrame,
    global_df: pd.DataFrame,
    missing_df: pd.DataFrame,
    notes_df: pd.DataFrame,
) -> None:
    os.makedirs(os.path.dirname(os.path.abspath(output_excel)), exist_ok=True)
    try:
        import openpyxl  # noqa: F401
        engine = "openpyxl"
    except Exception:
        engine = None

    with pd.ExcelWriter(output_excel, engine=engine) as writer:
        main_df.to_excel(writer, sheet_name="Bivariate_Table", index=False)
        pairwise_df.to_excel(writer, sheet_name="Pairwise_Test_Details", index=False)
        counts_df.to_excel(writer, sheet_name="Counts_By_Outcome", index=False)
        global_df.to_excel(writer, sheet_name="Global_Tests", index=False)
        missing_df.to_excel(writer, sheet_name="Missingness", index=False)
        notes_df.to_excel(writer, sheet_name="Notes", index=False)

        # Basic formatting when openpyxl is available.
        if engine == "openpyxl":
            from openpyxl.styles import Alignment, Font, PatternFill
            from openpyxl.utils import get_column_letter

            wb = writer.book
            header_fill = PatternFill("solid", fgColor="1F4E78")
            header_font = Font(color="FFFFFF", bold=True)

            for ws in wb.worksheets:
                ws.freeze_panes = "A2"
                ws.auto_filter.ref = ws.dimensions
                for cell in ws[1]:
                    cell.fill = header_fill
                    cell.font = header_font
                    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                for col_idx, col_cells in enumerate(ws.columns, start=1):
                    max_len = 0
                    for cell in col_cells:
                        val = "" if cell.value is None else str(cell.value)
                        max_len = max(max_len, len(val))
                    width = min(max(max_len + 2, 10), 45)
                    ws.column_dimensions[get_column_letter(col_idx)].width = width
                for row in ws.iter_rows():
                    for cell in row:
                        cell.alignment = Alignment(vertical="top", wrap_text=True)


def parse_sheet_arg(sheet_arg: str):
    try:
        return int(sheet_arg)
    except Exception:
        return sheet_arg


def parse_args():
    parser = argparse.ArgumentParser(
        description="Scenario 04 bivariate baseline analysis with home as reference category."
    )
    parser.add_argument("--input", default="/content/StrokeData_ML_dataset.xlsx", help="Input Excel file path")
    parser.add_argument("--sheet", default="0", help="Excel sheet name or zero-based sheet index")
    parser.add_argument(
        "--output-excel",
        default="/content/scenario04_bivariate_baseline_table.xlsx",
        help="Output Excel workbook path",
    )
    parser.add_argument("--alpha", type=float, default=0.05, help="Normality-test alpha level")
    parser.add_argument(
        "--fisher-mc-iter",
        type=int,
        default=20000,
        help="Monte Carlo draws for Fisher-Freeman-Halton approximation when a table is larger than 2x2",
    )
    parser.add_argument("--random-state", type=int, default=42, help="Random seed")

    # In Jupyter/Colab, the kernel may inject extra arguments such as:
    #   -f /root/.local/share/jupyter/runtime/kernel-....json
    # parse_known_args() prevents those unrelated kernel arguments from crashing
    # the analysis when the script is run with %run or pasted into a notebook cell.
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Ignoring non-script arguments injected by the notebook/kernel: {unknown}")
    return args


def main():
    args = parse_args()
    sheet = parse_sheet_arg(str(args.sheet))

    print("Reading input file...")
    df_raw = pd.read_excel(args.input, sheet_name=sheet)

    print("Standardizing columns and target labels using feature set...")
    data, target_col = standardize_dataframe(df_raw)

    class_counts = data[target_col].value_counts().reindex(TARGET_CLASSES).fillna(0).astype(int)
    print("Discharge class counts after cleaning:")
    print(class_counts.to_string())

    if class_counts.get(REFERENCE_CLASS, 0) == 0:
        raise ValueError("Reference class 'home' is absent after cleaning; pairwise home-reference tests cannot be run.")

    print("Running bivariate analyses...")
    main_df, pairwise_df, counts_df, global_df, missing_df = build_bivariate_tables(
        data=data,
        target_col=target_col,
        alpha=args.alpha,
        fisher_mc_iter=args.fisher_mc_iter,
        random_state=args.random_state,
    )
    notes_df = build_notes_df(args.input, len(data))

    print("Writing Excel output...")
    write_excel(
        output_excel=args.output_excel,
        main_df=main_df,
        pairwise_df=pairwise_df,
        counts_df=counts_df,
        global_df=global_df,
        missing_df=missing_df,
        notes_df=notes_df,
    )

    print("Done.")
    print(f"Excel output: {args.output_excel}")


if __name__ == "__main__":
    main()

#**ECDF of Age**

In [ ]:
from __future__ import annotations

import argparse
import os
import re
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


TARGET_CLASSES = ["home", "specialized care", "home with help", "expired"]
CLASS_LABELS = {
    "home": "Home (Class 0)",
    "specialized care": "Specialized Care (Class 1)",
    "home with help": "Home With Help (Class 2)",
    "expired": "Expired (Class 3)",
}
CLASS_COLORS = {
    "home": "blue",
    "specialized care": "green",
    "home with help": "red",
    "expired": "orange",
}

AGE_ALIASES = ["Age (Years)", "Age", "Patient Age", "age"]
TARGET_ALIASES = ["Discharge Disposition", "Discharge Disposition Summary", "Outcome", "discharge disposition"]

MISSING_STRINGS = {"", " ", "nan", "none", "null", "na", "n/a", "not avail", "not available", "unknown", "unk"}


def normalize_text(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    if s.lower() in MISSING_STRINGS:
        return np.nan
    return s


def find_column(df: pd.DataFrame, aliases: List[str]) -> Optional[str]:
    exact = {str(c).strip(): c for c in df.columns}
    lower = {str(c).strip().lower(): c for c in df.columns}
    for alias in aliases:
        if alias in exact:
            return exact[alias]
        if alias.lower() in lower:
            return lower[alias.lower()]
    return None


def normalize_target(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    sl = re.sub(r"\s+", " ", sl)

    if sl == "home":
        return "home"
    if sl in {"home with help", "home w help", "home w/ help", "home health", "home with home health", "home with assistance"}:
        return "home with help"
    if sl in {"specialized care", "snf", "rehab", "acute rehab", "skilled nursing", "ltac", "subacute rehab"}:
        return "specialized care"
    if "rehab" in sl or "snf" in sl or "nursing" in sl or "facility" in sl or "hospice" in sl:
        return "specialized care"
    if sl in {"expired", "death", "deceased", "dead", "died"}:
        return "expired"
    return np.nan


def empirical_cdf(values: np.ndarray):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = np.sort(values)
    if len(values) == 0:
        return values, values
    y = np.arange(1, len(values) + 1, dtype=float) / len(values)
    return values, y


def format_median_label(median_value: float) -> str:
    if np.isclose(median_value, round(median_value)):
        return f"{int(round(median_value))}"
    return f"{median_value:.1f}"


def plot_age_ecdf(
    df: pd.DataFrame,
    age_col: str,
    target_col: str,
    output_path: Optional[str] = None,
    dpi: int = 300,
    marker_size: float = 16.0,
    alpha: float = 0.85,
    x_min: float = 0.0,
    x_max: float = 105.0,
):
    work = df[[age_col, target_col]].copy()
    work[age_col] = pd.to_numeric(work[age_col], errors="coerce")
    work[target_col] = work[target_col].map(normalize_target)
    work = work.dropna(subset=[age_col, target_col])
    work = work[work[target_col].isin(TARGET_CLASSES)].copy()

    if work.empty:
        raise ValueError("No valid rows remained after cleaning Age and Discharge Disposition.")

    median_df = (
        work.groupby(target_col)[age_col]
        .median()
        .reindex(TARGET_CLASSES)
        .reset_index()
        .rename(columns={target_col: "Class", age_col: "Median Age"})
    )
    count_df = (
        work.groupby(target_col)[age_col]
        .size()
        .reindex(TARGET_CLASSES)
        .reset_index()
        .rename(columns={target_col: "Class", age_col: "N"})
    )
    median_df = median_df.merge(count_df, on="Class", how="left")
    median_df["Label"] = median_df["Class"].map(CLASS_LABELS)

    fig, ax = plt.subplots(figsize=(10, 7))

    for cls in TARGET_CLASSES:
        ages = work.loc[work[target_col] == cls, age_col].dropna().astype(float).to_numpy()
        if len(ages) == 0:
            continue

        x, y = empirical_cdf(ages)
        color = CLASS_COLORS[cls]
        label = CLASS_LABELS[cls]

        ax.scatter(
            x,
            y,
            s=marker_size,
            color=color,
            alpha=alpha,
            edgecolors="none",
            label=label,
        )

        median_age = float(median_df.loc[median_df["Class"] == cls, "Median Age"].iloc[0])

        ax.vlines(
            median_age,
            ymin=0,
            ymax=0.5,
            colors=color,
            linestyles="--",
            linewidth=1.4,
            alpha=0.8,
        )

        ax.text(
            median_age,
            -0.055,
            format_median_label(median_age),
            color=color,
            rotation=90,
            ha="center",
            va="top",
            fontsize=14,
            clip_on=False,
        )

    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1.2, alpha=0.8)

    ax.set_title("ECDF of Age by Discharge Disposition", fontsize=18, pad=12)
    ax.set_xlabel("Age (years)", fontsize=15)
    ax.set_ylabel("ECDF", fontsize=15)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_yticks(np.arange(0, 1.1, 0.1))
    ax.tick_params(axis="both", labelsize=13)
    ax.tick_params(axis="x", labelrotation=45)
    ax.legend(loc="upper left", fontsize=13, frameon=True)

    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

    plt.tight_layout()

    if output_path:
        os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
        fig.savefig(output_path, dpi=dpi, bbox_inches="tight")

    print("Rows used in ECDF:", len(work))
    print("Computed median age by discharge class (from the dataset):")
    for _, row in median_df.iterrows():
        if pd.notna(row["Median Age"]):
            print(f"  {row['Label']}: N={int(row['N'])}, median={row['Median Age']:.2f}")

    # Show exactly one figure in notebook/Colab output.
    # Using IPython.display.display(fig) plus an open matplotlib figure can make Colab render it twice.
    plt.show()
    plt.close(fig)

    return fig, median_df


def parse_sheet_arg(sheet_arg: str):
    try:
        return int(sheet_arg)
    except Exception:
        return sheet_arg


def main():
    parser = argparse.ArgumentParser(description="Plot ECDF of age by discharge disposition using medians computed from the input dataset.")
    parser.add_argument("--input", default="/content/StrokeData_ML_dataset.xlsx", help="Input Excel/CSV file path.")
    parser.add_argument("--sheet", default="0", help="Excel sheet name or zero-based sheet index. Ignored for CSV.")
    parser.add_argument("--output", default="/content/ecdf_age_by_discharge.png", help="Output PNG path. Use --no-save to disable saving.")
    parser.add_argument("--no-save", action="store_true", help="Do not save the figure; only display it.")
    parser.add_argument("--dpi", type=int, default=300, help="Figure DPI for saved image.")
    parser.add_argument("--marker-size", type=float, default=16.0, help="Point size in the ECDF plot.")
    parser.add_argument("--alpha", type=float, default=0.85, help="Point transparency.")
    args, _unknown = parser.parse_known_args()

    input_path = args.input
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input file not found: {input_path}")

    ext = os.path.splitext(input_path)[1].lower()
    if ext in [".xlsx", ".xls", ".xlsm"]:
        df = pd.read_excel(input_path, sheet_name=parse_sheet_arg(str(args.sheet)))
    elif ext in [".csv", ".txt"]:
        df = pd.read_csv(input_path)
    else:
        raise ValueError("Input file must be .xlsx, .xls, .xlsm, .csv, or .txt")

    age_col = find_column(df, AGE_ALIASES)
    target_col = find_column(df, TARGET_ALIASES)

    if age_col is None:
        raise ValueError(f"Age column was not found. Expected one of: {AGE_ALIASES}")
    if target_col is None:
        raise ValueError(f"Discharge disposition column was not found. Expected one of: {TARGET_ALIASES}")

    output_path = None if args.no_save else args.output

    plot_age_ecdf(
        df=df,
        age_col=age_col,
        target_col=target_col,
        output_path=output_path,
        dpi=args.dpi,
        marker_size=args.marker_size,
        alpha=args.alpha,
    )


if __name__ == "__main__":
    main()

#**3D PCA**

In [ ]:
from __future__ import annotations

import argparse
import os
import re
import warnings
from typing import List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

TARGET_CLASSES = ["home", "specialized care", "home with help", "expired"]
TARGET_DISPLAY = {
    "home": "Home",
    "specialized care": "Specialized Care",
    "home with help": "Home with Help",
    "expired": "Expired",
}

FEATURE_ALIASES = {
    "Sex": ["Sex"],
    "Last Known Well": ["Last Known Well", "Final Last Known Well"],
    "Smoking Status": ["Smoking Status"],
    "Aspirin Use": ["Aspirin Use", "Aspirin x=yes"],
    "tPA administered": [
        "tPA administered",
        "tPA administered?",
        "tPA (alteplase) administered",
        "Alteplase administered",
        "tPA (alteplase) administered? May be listed as alteplase in Neurology consult. With or without later bleed.",
    ],
    "Drug Screen Positive": [
        "Drug Screen Positive",
        "Lab Evidence of Drug Panel (urine tox screen) Often Ethanol appears too but not looking at this per se",
    ],
    "Primary Diagnosis": ["Primary Diagnosis", "Enc - Primary ICD10 Diagnosis (Billing Code)"],
    "Race": ["Race"],
    "Previous Stroke": ["Previous Stroke", "Evidence of Previous Stroke"],
    "Depression": ["Depression", "Depression x=yes"],
    "Insurance Type": ["Insurance Type", "Insurance C=Commercial, M=Medicare, A=Medicaid, O=Other"],
    "Service": ["Service", "Service (Y/N)", "Service Number = Underserved, No = Served"],
    "Troponin": [
        "Troponin",
        "Troponin 1",
        "Troponin 1 (N;Normal; (less than 0.05 ng/mL), H;High)",
    ],
    "LOS (Days)": ["LOS (Days)", "Hospital Stay (Days)", "Calendar Days in Hospital"],
    "Age (Years)": ["Age (Years)", "Age"],
    "LDL (mg/dL)": ["LDL (mg/dL)"],
    "Systolic BP (mmHg)": ["Systolic BP (mmHg)", "Systolic Extraction"],
    "Blood Glucose (mg/dL)": ["Blood Glucose (mg/dL)"],
    "Admission NIHSS": ["Admission NIHSS", "Final NIHSS", "NIHSS"],
}

ANTICOAGULANT_ALIASES = ["Anticoagulant", "Anticoagulant x=yes", "Anticoagulant x = yes"]
TPA_BLEED_ALIASES = [
    "Evidence of Bleed following tPA? If No tPA state NA ",
    "Evidence of Bleed following tPA? If No tPA state NA",
    "Evidence of Bleed following tPA",
    "Bleed following tPA",
]
TARGET_ALIASES = ["Discharge Disposition", "Discharge Disposition Summary"]

NUMERIC_FEATURES = [
    "LOS (Days)",
    "Age (Years)",
    "LDL (mg/dL)",
    "Systolic BP (mmHg)",
    "Blood Glucose (mg/dL)",
    "Admission NIHSS",
]

ORDINAL_FEATURES = ["Last Known Well", "Smoking Status", "Troponin"]
BINARY_FEATURES = [
    "Sex",
    "Aspirin Use",
    "Anticoagulant",
    "Drug Screen Positive",
    "Previous Stroke",
    "Depression",
    "Service",
]
NOMINAL_FEATURES = ["Primary Diagnosis", "Race", "Insurance Type", "tPA administered"]

ORDINAL_CATEGORIES = {
    "Last Known Well": ["< 3 hours", "3 to 6 hours", "6 to 12 hours", "12-24 hours", "> 24 hours"],
    "Smoking Status": ["Never", "Former", "Current"],
    "Troponin": ["Normal", "High"],
}

BINARY_CATEGORIES = {
    "Sex": ["Female", "Male"],
    "Aspirin Use": ["No", "Yes"],
    "Anticoagulant": ["No", "Yes"],
    "Drug Screen Positive": ["No", "Yes"],
    "Previous Stroke": ["No", "Yes"],
    "Depression": ["No", "Yes"],
    "Service": ["No", "Yes"],
}

ALL_FEATURES = (
    ["Sex", "Last Known Well", "Smoking Status", "Aspirin Use", "Anticoagulant", "tPA administered", "Drug Screen Positive"]
    + ["Primary Diagnosis", "Race", "Previous Stroke", "Depression", "Insurance Type", "Service", "Troponin"]
    + NUMERIC_FEATURES
)

MISSING_STRINGS = {"", " ", "nan", "none", "null", "na", "n/a", "not avail", "not available", "unknown", "unk"}


def normalize_text(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    if s.lower() in MISSING_STRINGS:
        return np.nan
    return s


def find_column(df: pd.DataFrame, aliases: List[str]) -> Optional[str]:
    exact = {str(c).strip(): c for c in df.columns}
    lower = {str(c).strip().lower(): c for c in df.columns}
    for alias in aliases:
        if alias in exact:
            return exact[alias]
        if alias.lower() in lower:
            return lower[alias.lower()]
    return None


def normalize_yes_no(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    yes_values = {"yes", "y", "true", "1", "x", "positive", "pos", "+", "present", "underserved"}
    no_values = {"no", "n", "false", "0", "negative", "neg", "-", "absent", "served"}
    if sl in yes_values:
        return "Yes"
    if sl in no_values:
        return "No"
    return np.nan


def normalize_tpa_administered(tpa_value, bleed_value=None):
    tpa = normalize_text(tpa_value)
    bleed = normalize_text(bleed_value) if bleed_value is not None else np.nan
    tpa_l = str(tpa).strip().lower() if not pd.isna(tpa) else ""
    bleed_l = str(bleed).strip().lower() if not pd.isna(bleed) else ""

    if tpa_l in {"no", "n", "0", "false", "f", "not given", "none"}:
        return "No"
    if tpa_l in {"yes - no bleed", "yes-no bleed", "yes no bleed", "yes without bleed", "yes - without bleed", "no bleed"}:
        return "Yes - No bleed"
    if tpa_l in {"yes - with bleed", "yes-with bleed", "yes with bleed", "yes and bleed", "yes - bleed", "with bleed", "bleed"}:
        return "Yes - With bleed"
    if tpa_l in {"yes", "y", "1", "true", "t", "alteplase", "tpa", "t-pa"}:
        if bleed_l in {"yes", "y", "1", "true", "t", "+", "positive", "pos", "present", "with bleed", "bleed"}:
            return "Yes - With bleed"
        if bleed_l in {"no", "n", "0", "false", "f", "-", "negative", "neg", "absent", "no bleed"}:
            return "Yes - No bleed"
        return np.nan
    if tpa_l == "":
        if bleed_l in {"yes", "y", "1", "true", "t", "+", "positive", "pos", "present", "with bleed", "bleed"}:
            return "Yes - With bleed"
        if bleed_l in {"no", "n", "0", "false", "f", "-", "negative", "neg", "absent", "no bleed"}:
            return "Yes - No bleed"
        return np.nan
    return np.nan


def normalize_sex(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"f", "female", "woman"}:
        return "Female"
    if sl in {"m", "male", "man"}:
        return "Male"
    return np.nan


def normalize_lkw(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).lower().replace("hrs", "hours").replace("hour", "hours").strip()
    sl = re.sub(r"\s+", " ", sl)
    if "<" in sl and "3" in sl:
        return "< 3 hours"
    if "less" in sl and "3" in sl:
        return "< 3 hours"
    if ">" in sl and "24" in sl:
        return "> 24 hours"
    if "greater" in sl and "24" in sl:
        return "> 24 hours"
    if ("3" in sl and "6" in sl) or sl in {"3-6", "3 to 6"}:
        return "3 to 6 hours"
    if ("6" in sl and "12" in sl) or sl in {"6-12", "6 to 12"}:
        return "6 to 12 hours"
    if ("12" in sl and "24" in sl) or sl in {"12-24", "12 to 24"}:
        return "12-24 hours"
    return np.nan


def normalize_smoking(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"never", "no", "none", "non-smoker", "nonsmoker", "never smoker"}:
        return "Never"
    if sl in {"former", "previous", "past", "ex-smoker", "ex smoker", "former smoker"}:
        return "Former"
    if sl in {"current", "yes", "active", "smoker", "current smoker"}:
        return "Current"
    return np.nan


def normalize_diagnosis(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).lower()
    if "tia" in sl or "transient" in sl:
        return "TIA"
    if "hemorr" in sl or "haemorr" in sl or "ich" in sl or "sah" in sl or "bleed" in sl:
        return "Hemorrhage"
    if "ischem" in sl or "ischaem" in sl or "infarct" in sl or sl.startswith("i"):
        return "Ischemic"
    return np.nan


def normalize_race(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).lower()
    if "white" in sl or "cauc" in sl:
        return "White"
    if "black" in sl or "african" in sl:
        return "Black"
    if "hispanic" in sl or "latino" in sl or "latina" in sl:
        return "Hispanic"
    if "asian" in sl:
        return "Asian"
    return "Other"


def normalize_insurance(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"c", "commercial", "private"} or "commercial" in sl:
        return "Commercial"
    if sl in {"m", "medicare"} or "medicare" in sl:
        return "Medicare"
    if sl in {"a", "medicaid"} or "medicaid" in sl:
        return "Medicaid"
    if sl in {"o", "other", "self pay", "self-pay", "uninsured"}:
        return "Other"
    return "Other"


def normalize_troponin(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"n", "normal", "negative", "neg", "0"}:
        return "Normal"
    if sl in {"h", "high", "positive", "pos", "1", "elevated", "abnormal"}:
        return "High"
    cleaned = sl.replace("ng/ml", "")
    if "less" in cleaned or "<" in cleaned:
        nums = re.findall(r"[-+]?\d*\.?\d+", cleaned)
        if nums:
            try:
                if float(nums[0]) <= 0.05:
                    return "Normal"
            except Exception:
                pass
    nums = re.findall(r"[-+]?\d*\.?\d+", cleaned)
    if nums:
        try:
            val = float(nums[0])
            return "High" if val >= 0.05 else "Normal"
        except Exception:
            return np.nan
    return np.nan


def normalize_target(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    sl = re.sub(r"\s+", " ", sl)
    if sl == "home":
        return "home"
    if sl in {"home with help", "home w help", "home w/ help", "home health", "home with home health", "home with assistance"}:
        return "home with help"
    if sl in {"specialized care", "snf", "rehab", "acute rehab", "skilled nursing", "ltac", "subacute rehab"}:
        return "specialized care"
    if "rehab" in sl or "snf" in sl or "nursing" in sl or "facility" in sl or "hospice" in sl:
        return "specialized care"
    if sl in {"expired", "death", "deceased", "dead", "died"}:
        return "expired"
    return np.nan


def standardize_raw_dataframe(df_raw: pd.DataFrame, require_anticoagulant: bool = True) -> Tuple[pd.DataFrame, str]:
    aliases = dict(FEATURE_ALIASES)
    aliases["Anticoagulant"] = ANTICOAGULANT_ALIASES

    data = pd.DataFrame(index=df_raw.index)
    missing_cols = []
    for canonical, col_aliases in aliases.items():
        src = find_column(df_raw, col_aliases)
        if src is None:
            if canonical == "Anticoagulant" and not require_anticoagulant:
                data[canonical] = np.nan
            else:
                missing_cols.append(canonical)
        elif canonical == "tPA administered":
            bleed_src = find_column(df_raw, TPA_BLEED_ALIASES)
            if bleed_src is None:
                data[canonical] = df_raw[src].map(lambda value: normalize_tpa_administered(value, None))
            else:
                data[canonical] = [
                    normalize_tpa_administered(tpa_value, bleed_value)
                    for tpa_value, bleed_value in zip(df_raw[src], df_raw[bleed_src])
                ]
        else:
            data[canonical] = df_raw[src]

    target_src = find_column(df_raw, TARGET_ALIASES)
    if target_src is None:
        raise ValueError(f"Target column not found. Expected one of: {TARGET_ALIASES}")
    data["Discharge Disposition"] = df_raw[target_src]

    if missing_cols:
        raise ValueError("Missing required feature columns: " + ", ".join(missing_cols))

    for col in NUMERIC_FEATURES:
        data[col] = pd.to_numeric(data[col].replace(list(MISSING_STRINGS), np.nan), errors="coerce")

    data["Sex"] = data["Sex"].map(normalize_sex)
    data["Last Known Well"] = data["Last Known Well"].map(normalize_lkw)
    data["Smoking Status"] = data["Smoking Status"].map(normalize_smoking)
    for col in [c for c in BINARY_FEATURES if c != "Sex"]:
        data[col] = data[col].map(normalize_yes_no)
    data["Primary Diagnosis"] = data["Primary Diagnosis"].map(normalize_diagnosis)
    data["Race"] = data["Race"].map(normalize_race)
    data["Insurance Type"] = data["Insurance Type"].map(normalize_insurance)
    data["Troponin"] = data["Troponin"].map(normalize_troponin)
    data["Discharge Disposition"] = data["Discharge Disposition"].map(normalize_target)

    before = len(data)
    data = data[data["Discharge Disposition"].isin(TARGET_CLASSES)].copy()
    dropped = before - len(data)
    if dropped:
        print(f"Dropped {dropped} rows because discharge outcome was missing/outside target classes.")

    return data, "Discharge Disposition"


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocess_pipeline() -> Pipeline:
    ordinal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            categories=[ORDINAL_CATEGORIES[col] for col in ORDINAL_FEATURES],
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )),
    ])

    binary_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            categories=[BINARY_CATEGORIES[col] for col in BINARY_FEATURES],
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )),
    ])

    nominal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("encoder", make_one_hot_encoder()),
    ])

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipe, NUMERIC_FEATURES),
            ("ordinal", ordinal_pipe, ORDINAL_FEATURES),
            ("binary", binary_pipe, BINARY_FEATURES),
            ("nominal", nominal_pipe, NOMINAL_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    return Pipeline([
        ("preprocess", preprocessor),
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=3, random_state=42)),
    ])


def plot_3d_pca(scores_df: pd.DataFrame, elev: float = 24.0, azim: float = -63.0, marker_size: float = 180.0):
    from matplotlib.colors import BoundaryNorm

    class_to_code = {cls: i for i, cls in enumerate(TARGET_CLASSES)}
    codes = scores_df["Discharge Disposition"].map(class_to_code).to_numpy()

    fig = plt.figure(figsize=(11, 8.5))
    ax = fig.add_subplot(111, projection="3d")

    cmap = plt.get_cmap("viridis", len(TARGET_CLASSES))
    bounds = np.arange(-0.5, len(TARGET_CLASSES) + 0.5, 1)
    norm = BoundaryNorm(bounds, cmap.N)

    scatter = ax.scatter(
        scores_df["Principal Component 1"],
        scores_df["Principal Component 2"],
        scores_df["Principal Component 3"],
        c=codes,
        cmap=cmap,
        norm=norm,
        s=marker_size,
        alpha=0.85,
        edgecolors="black",
        linewidths=0.35,
    )

    ax.set_xlabel("Principal Component 1", labelpad=10, fontsize=12)
    ax.set_ylabel("Principal Component 2", labelpad=10, fontsize=12)
    ax.set_zlabel("Principal Component 3", labelpad=10, fontsize=12)
    ax.view_init(elev=elev, azim=azim)
    ax.grid(True)

    cbar = fig.colorbar(scatter, ax=ax, fraction=0.035, pad=0.08, ticks=list(range(len(TARGET_CLASSES))))
    cbar.ax.set_yticklabels([TARGET_DISPLAY[c] for c in TARGET_CLASSES])
    cbar.ax.tick_params(labelsize=12)

    plt.title("3D PCA of Raw Pre-split Cohort by Discharge Outcome", pad=18, fontsize=14)
    plt.tight_layout()
    return fig


def main():
    parser = argparse.ArgumentParser(description="Display a pre-train/test 3D PCA plot from raw stroke dataset with tPA administered included.")
    parser.add_argument("--input", default="StrokeData_ML_dataset.xlsx", help="Input Excel/CSV file path.")
    parser.add_argument("--sheet", default=0, help="Excel sheet name or index. Ignored for CSV.")
    parser.add_argument("--require-anticoagulant", action="store_true", default=True, help="Require Anticoagulant column; default True.")
    parser.add_argument("--allow-missing-anticoagulant", action="store_true", help="Proceed even if Anticoagulant column is absent.")
    parser.add_argument("--elev", type=float, default=24.0, help="3D plot elevation angle.")
    parser.add_argument("--azim", type=float, default=-63.0, help="3D plot azimuth angle.")
    parser.add_argument("--marker-size", type=float, default=80.0, help="Scatter marker size.")
    args, _unknown = parser.parse_known_args()

    input_path = args.input
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input file not found: {input_path}")

    ext = os.path.splitext(input_path)[1].lower()
    if ext in [".xlsx", ".xls", ".xlsm"]:
        sheet = args.sheet
        try:
            sheet = int(sheet)
        except Exception:
            pass
        df_raw = pd.read_excel(input_path, sheet_name=sheet)
    elif ext in [".csv", ".txt"]:
        df_raw = pd.read_csv(input_path)
    else:
        raise ValueError("Input file must be .xlsx, .xls, .xlsm, .csv, or .txt")

    require_anticoagulant = bool(args.require_anticoagulant and not args.allow_missing_anticoagulant)
    data, target_col = standardize_raw_dataframe(df_raw, require_anticoagulant=require_anticoagulant)

    X_raw = data[ALL_FEATURES].copy()
    y = data[target_col].copy()

    pca_pipe = build_preprocess_pipeline()
    pca_scores = pca_pipe.fit_transform(X_raw)
    scores_df = pd.DataFrame({
        "Original Row Index": data.index,
        "Discharge Disposition": y.to_numpy(),
        "Principal Component 1": pca_scores[:, 0],
        "Principal Component 2": pca_scores[:, 1],
        "Principal Component 3": pca_scores[:, 2],
    })

    print("Number of rows in scores_df:", len(scores_df))
    print("Number of PCA points:", pca_scores.shape[0])

    fig = plot_3d_pca(scores_df, elev=args.elev, azim=args.azim, marker_size=args.marker_size)

    # Show the figure only once.
    # In Jupyter/Colab, using IPython.display.display(fig) can render the same
    # open Matplotlib figure again through the inline backend at the end of the cell.
    plt.show()
    plt.close(fig)


if __name__ == "__main__":
    main()

#**Final Round of MLP Hyperparameter Tuning: Narrow Parameter Range**

In [ ]:
"""
The final round of MLP hyperparameter tuning utilized a narrowed range of parameters.
"""

from __future__ import annotations

import argparse
import inspect
import os
import re
import subprocess
import sys
import warnings
from typing import Dict, List, Optional, Tuple

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import (
    accuracy_score,
    auc,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, learning_curve, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, RobustScaler, StandardScaler, label_binarize
from sklearn.utils import check_random_state
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# -----------------------------------------------------------------------------
# Modeling variables
# -----------------------------------------------------------------------------

TARGET_CLASSES = ["home", "specialized care", "home with help", "expired"]

FEATURE_ALIASES = {
    "Sex": ["Sex"],
    "Last Known Well": ["Last Known Well", "Final Last Known Well"],
    "Smoking Status": ["Smoking Status"],
    "Aspirin Use": ["Aspirin Use", "Aspirin x=yes"],
    "Drug Screen Positive": [
        "Drug Screen Positive",
        "Lab Evidence of Drug Panel (urine tox screen) Often Ethanol appears too but not looking at this per se",
    ],
    "Primary Diagnosis": ["Primary Diagnosis", "Enc - Primary ICD10 Diagnosis (Billing Code)"],
    "Race": ["Race"],
    "Previous Stroke": ["Previous Stroke", "Evidence of Previous Stroke"],
    "Depression": ["Depression", "Depression x=yes"],
    "Insurance Type": ["Insurance Type", "Insurance C=Commercial, M=Medicare, A=Medicaid, O=Other"],
    "Service": ["Service", "Service (Y/N)", "Service Number = Underserved, No = Served"],
    "tPA administered": [
        "tPA administered",
        "tPA Administered",
        "TPA administered",
        "TPA Administered",
        "tPA",
        "TPA",
    ],
    "Troponin": [
        "Troponin",
        "Troponin 1",
        "Troponin 1 (N;Normal; (less than 0.05 ng/mL), H;High)",
    ],
    "LOS (Days)": ["LOS (Days)", "Hospital Stay (Days)", "Calendar Days in Hospital"],
    "Age (Years)": ["Age (Years)", "Age"],
    "LDL (mg/dL)": ["LDL (mg/dL)"],
    "Systolic BP (mmHg)": ["Systolic BP (mmHg)", "Systolic Extraction"],
    "Blood Glucose (mg/dL)": ["Blood Glucose (mg/dL)"],
    "Admission NIHSS": ["Admission NIHSS", "Final NIHSS"],
}

TARGET_ALIASES = ["Discharge Disposition", "Discharge Disposition Summary"]

NUMERIC_FEATURES = [
    "LOS (Days)",
    "Age (Years)",
    "LDL (mg/dL)",
    "Systolic BP (mmHg)",
    "Blood Glucose (mg/dL)",
    "Admission NIHSS",
]

# True ordered clinical features. These are ordinal-encoded and not one-hot encoded.
ORDINAL_FEATURES = [
    "Last Known Well",
    "Smoking Status",
    "Troponin",
]

# Binary features. These are 0/1 encoded and not one-hot encoded.
BINARY_FEATURES = [
    "Sex",
    "Aspirin Use",
    "Drug Screen Positive",
    "Previous Stroke",
    "Depression",
    "Service",
]

# Truly nominal categorical features. These are one-hot encoded.
NOMINAL_FEATURES = ["Primary Diagnosis", "Race", "Insurance Type", "tPA administered"]

ORDINAL_CATEGORIES = {
    "Last Known Well": ["< 3 hours", "3 to 6 hours", "6 to 12 hours", "12-24 hours", "> 24 hours"],
    "Smoking Status": ["Never", "Former", "Current"],
    "Troponin": ["Normal", "High"],
}

BINARY_CATEGORIES = {
    "Sex": ["Female", "Male"],
    "Aspirin Use": ["No", "Yes"],
    "Drug Screen Positive": ["No", "Yes"],
    "Previous Stroke": ["No", "Yes"],
    "Depression": ["No", "Yes"],
    "Service": ["No", "Yes"],
}

KNOWN_NOMINAL_CATEGORIES = {
    "Primary Diagnosis": ["Ischemic", "TIA", "Hemorrhage"],
    "Race": ["White", "Black", "Hispanic", "Asian", "Other"],
    "Insurance Type": ["Commercial", "Medicare", "Medicaid", "Other"],
    "tPA administered": ["No", "Yes - No bleed", "Yes - With bleed"],
}

ALL_FEATURES = list(FEATURE_ALIASES.keys())

# Additional binary predictor aliases.
ANTICOAGULANT_ALIASES = ["Anticoagulant", "Anticoagulant x=yes", "Anticoagulant x = yes"]
BASE_FEATURE_ALIASES = dict(FEATURE_ALIASES)
BASE_BINARY_FEATURES = list(BINARY_FEATURES)
BASE_BINARY_CATEGORIES = dict(BINARY_CATEGORIES)

def configure_feature_set(include_anticoagulant: bool) -> None:
    """Configure the final prespecified feature set."""
    global FEATURE_ALIASES, BINARY_FEATURES, BINARY_CATEGORIES, ALL_FEATURES

    FEATURE_ALIASES = dict(BASE_FEATURE_ALIASES)
    BINARY_FEATURES = list(BASE_BINARY_FEATURES)
    BINARY_CATEGORIES = dict(BASE_BINARY_CATEGORIES)

    if include_anticoagulant:
        FEATURE_ALIASES["Anticoagulant"] = ANTICOAGULANT_ALIASES
        if "Anticoagulant" not in BINARY_FEATURES:
            insert_after = BINARY_FEATURES.index("Aspirin Use") + 1 if "Aspirin Use" in BINARY_FEATURES else len(BINARY_FEATURES)
            BINARY_FEATURES.insert(insert_after, "Anticoagulant")
        BINARY_CATEGORIES["Anticoagulant"] = ["No", "Yes"]

    ALL_FEATURES = list(FEATURE_ALIASES.keys())

MISSING_STRINGS = {"", " ", "nan", "none", "null", "na", "n/a", "not avail", "not available", "unknown", "unk"}

# -----------------------------------------------------------------------------
# Cleaning and standardization
# -----------------------------------------------------------------------------

def normalize_text(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    if s.lower() in MISSING_STRINGS:
        return np.nan
    return s


def find_column(df: pd.DataFrame, aliases: List[str]) -> Optional[str]:
    exact = {str(c).strip(): c for c in df.columns}
    lower = {str(c).strip().lower(): c for c in df.columns}
    for alias in aliases:
        if alias in exact:
            return exact[alias]
        if alias.lower() in lower:
            return lower[alias.lower()]
    return None


def normalize_yes_no(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    yes_values = {"yes", "y", "true", "1", "x", "positive", "pos", "+", "present", "underserved"}
    no_values = {"no", "n", "false", "0", "negative", "neg", "-", "absent", "served"}
    if sl in yes_values:
        return "Yes"
    if sl in no_values:
        return "No"
    return np.nan


def normalize_sex(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"f", "female", "woman"}:
        return "Female"
    if sl in {"m", "male", "man"}:
        return "Male"
    return np.nan


def normalize_lkw(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).lower().replace("hrs", "hours").replace("hour", "hours").strip()
    sl = re.sub(r"\s+", " ", sl)
    if "<" in sl and "3" in sl:
        return "< 3 hours"
    if "less" in sl and "3" in sl:
        return "< 3 hours"
    if ">" in sl and "24" in sl:
        return "> 24 hours"
    if "greater" in sl and "24" in sl:
        return "> 24 hours"
    if ("3" in sl and "6" in sl) or sl in {"3-6", "3 to 6"}:
        return "3 to 6 hours"
    if ("6" in sl and "12" in sl) or sl in {"6-12", "6 to 12"}:
        return "6 to 12 hours"
    if ("12" in sl and "24" in sl) or sl in {"12-24", "12 to 24"}:
        return "12-24 hours"
    return np.nan


def normalize_smoking(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"never", "no", "none", "non-smoker", "nonsmoker", "never smoker"}:
        return "Never"
    if sl in {"former", "previous", "past", "ex-smoker", "ex smoker", "former smoker"}:
        return "Former"
    if sl in {"current", "yes", "active", "smoker", "current smoker"}:
        return "Current"
    return np.nan


def normalize_diagnosis(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).lower()
    if "tia" in sl or "transient" in sl:
        return "TIA"
    if "hemorr" in sl or "haemorr" in sl or "ich" in sl or "sah" in sl or "bleed" in sl:
        return "Hemorrhage"
    if "ischem" in sl or "ischaem" in sl or "infarct" in sl or sl.startswith("i"):
        return "Ischemic"
    return np.nan


def normalize_race(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).lower()
    if "white" in sl or "cauc" in sl:
        return "White"
    if "black" in sl or "african" in sl:
        return "Black"
    if "hispanic" in sl or "latino" in sl or "latina" in sl:
        return "Hispanic"
    if "asian" in sl:
        return "Asian"
    return "Other"


def normalize_insurance(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"c", "commercial", "private"} or "commercial" in sl:
        return "Commercial"
    if sl in {"m", "medicare"} or "medicare" in sl:
        return "Medicare"
    if sl in {"a", "medicaid"} or "medicaid" in sl:
        return "Medicaid"
    if sl in {"o", "other", "self pay", "self-pay", "uninsured"}:
        return "Other"
    return "Other"


def normalize_troponin(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    if sl in {"n", "normal", "negative", "neg", "0"}:
        return "Normal"
    if sl in {"h", "high", "positive", "pos", "1", "elevated", "abnormal"}:
        return "High"
    # Handle possible strings like "<0.05", "less than 0.05", or numeric values.
    cleaned = sl.replace("ng/ml", "").replace("ng/mL".lower(), "")
    if "less" in cleaned or "<" in cleaned:
        nums = re.findall(r"[-+]?\d*\.?\d+", cleaned)
        if nums:
            try:
                if float(nums[0]) <= 0.05:
                    return "Normal"
            except Exception:
                pass
    nums = re.findall(r"[-+]?\d*\.?\d+", cleaned)
    if nums:
        try:
            val = float(nums[0])
            return "High" if val >= 0.05 else "Normal"
        except Exception:
            return np.nan
    return np.nan



def normalize_tpa_administered(x):
    """
    Three-level thrombolysis-treatment feature.

    Allowed output levels:
        No
        Yes - No bleed
        Yes - With bleed

    This is treated as a nominal categorical feature, not as a binary feature,
    because it has three clinically distinct levels.
    """
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    sl = re.sub(r"\s+", " ", sl)

    no_values = {"no", "n", "0", "false", "not given", "not administered", "none", "negative"}
    yes_no_bleed_values = {
        "yes - no bleed", "yes-no bleed", "yes no bleed", "yes without bleed",
        "yes - without bleed", "yes without hemorrhage", "yes - no hemorrhage",
        "yes no hemorrhage", "tpa no bleed", "tpa without bleed", "1"
    }
    yes_with_bleed_values = {
        "yes - with bleed", "yes-with bleed", "yes with bleed", "yes with hemorrhage",
        "yes - with hemorrhage", "tpa with bleed", "tpa with hemorrhage", "bleed", "hemorrhage"
    }

    if sl in no_values:
        return "No"
    if sl in yes_no_bleed_values:
        return "Yes - No bleed"
    if sl in yes_with_bleed_values:
        return "Yes - With bleed"

    if "yes" in sl or "tpa" in sl or "alteplase" in sl or "thrombolysis" in sl:
        if "with" in sl and ("bleed" in sl or "hemorr" in sl or "haemorr" in sl):
            return "Yes - With bleed"
        if "bleed" in sl or "hemorr" in sl or "haemorr" in sl:
            if "no" in sl or "without" in sl:
                return "Yes - No bleed"
            return "Yes - With bleed"
        return "Yes - No bleed"

    return np.nan

def normalize_target(x):
    s = normalize_text(x)
    if pd.isna(s):
        return np.nan
    sl = str(s).strip().lower()
    sl = re.sub(r"\s+", " ", sl)
    if sl == "home":
        return "home"
    if sl in {"home with help", "home w help", "home w/ help", "home health", "home with home health"}:
        return "home with help"
    if sl in {"specialized care", "snf", "rehab", "acute rehab", "skilled nursing", "ltac", "subacute rehab"}:
        return "specialized care"
    if "rehab" in sl or "snf" in sl or "nursing" in sl or "facility" in sl or "hospice" in sl:
        return "specialized care"
    if sl in {"expired", "death", "deceased", "dead", "died"}:
        return "expired"
    return np.nan


def standardize_dataframe(df_raw: pd.DataFrame) -> Tuple[pd.DataFrame, str]:
    data = pd.DataFrame(index=df_raw.index)

    missing_cols = []
    for canonical, aliases in FEATURE_ALIASES.items():
        src = find_column(df_raw, aliases)
        if src is None:
            missing_cols.append(canonical)
        else:
            data[canonical] = df_raw[src]

    target_src = find_column(df_raw, TARGET_ALIASES)
    if target_src is None:
        raise ValueError(f"Target column was not found. Expected one of: {TARGET_ALIASES}")
    data["Discharge Disposition"] = df_raw[target_src]

    if missing_cols:
        raise ValueError("The following requested feature columns were not found: " + ", ".join(missing_cols))

    for col in NUMERIC_FEATURES:
        data[col] = pd.to_numeric(data[col].replace(list(MISSING_STRINGS), np.nan), errors="coerce")

    data["Sex"] = data["Sex"].map(normalize_sex)
    data["Last Known Well"] = data["Last Known Well"].map(normalize_lkw)
    data["Smoking Status"] = data["Smoking Status"].map(normalize_smoking)
    for col in [c for c in BINARY_FEATURES if c != "Sex"]:
        data[col] = data[col].map(normalize_yes_no)
    data["Primary Diagnosis"] = data["Primary Diagnosis"].map(normalize_diagnosis)
    data["Race"] = data["Race"].map(normalize_race)
    data["Insurance Type"] = data["Insurance Type"].map(normalize_insurance)
    data["tPA administered"] = data["tPA administered"].map(normalize_tpa_administered)
    data["Troponin"] = data["Troponin"].map(normalize_troponin)
    data["Discharge Disposition"] = data["Discharge Disposition"].map(normalize_target)

    before = len(data)
    data = data[data["Discharge Disposition"].isin(TARGET_CLASSES)].copy()
    dropped = before - len(data)
    if dropped > 0:
        print(f"Dropped {dropped} rows because target was missing or outside the 4 requested classes.")

    return data, "Discharge Disposition"

# -----------------------------------------------------------------------------
# Mixed-type MICE imputer
# -----------------------------------------------------------------------------

class MixedTypeMICEImputer(BaseEstimator, TransformerMixin):
    """Approximate MICE for mixed numeric + categorical data inside CV pipeline."""

    def __init__(self, numeric_columns=None, categorical_columns=None, known_categories=None,
                 max_iter=30, random_state=42):
        self.numeric_columns = numeric_columns
        self.categorical_columns = categorical_columns
        self.known_categories = known_categories
        self.max_iter = max_iter
        self.random_state = random_state

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columns_ = list(X.columns)
        self.numeric_columns_ = list(self.numeric_columns or [])
        self.categorical_columns_ = list(self.categorical_columns or [])
        self.category_maps_ = {}
        self.inverse_category_maps_ = {}

        for col in self.categorical_columns_:
            observed = [v for v in pd.Series(X[col]).dropna().astype(str).unique().tolist()]
            known = []
            if self.known_categories and col in self.known_categories:
                known = list(self.known_categories[col])
            categories = []
            for v in known + sorted(observed):
                if v not in categories:
                    categories.append(v)
            if not categories:
                categories = ["Unknown"]
            self.category_maps_[col] = {cat: i for i, cat in enumerate(categories)}
            self.inverse_category_maps_[col] = {i: cat for cat, i in self.category_maps_[col].items()}

        matrix = self._to_numeric_matrix(X)
        self.all_missing_columns_ = np.where(np.all(pd.isna(matrix), axis=0))[0].tolist()
        if self.all_missing_columns_:
            matrix[:, self.all_missing_columns_] = 0.0

        self.imputer_ = IterativeImputer(
            max_iter=self.max_iter,
            random_state=self.random_state,
            initial_strategy="most_frequent",
            sample_posterior=False,
            skip_complete=True,
        )
        self.imputer_.fit(matrix)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        matrix = self._to_numeric_matrix(X)
        if getattr(self, "all_missing_columns_", None):
            matrix[:, self.all_missing_columns_] = 0.0
        imputed = self.imputer_.transform(matrix)
        return self._from_numeric_matrix(imputed, index=X.index)

    def _to_numeric_matrix(self, X: pd.DataFrame) -> np.ndarray:
        cols = []
        for col in self.columns_:
            if col in self.numeric_columns_:
                vals = pd.to_numeric(X[col], errors="coerce").astype(float).to_numpy()
            else:
                mapping = self.category_maps_[col]
                vals = X[col].map(mapping).astype(float).to_numpy()
            cols.append(vals)
        return np.column_stack(cols).astype(float)

    def _from_numeric_matrix(self, matrix: np.ndarray, index=None) -> pd.DataFrame:
        out = pd.DataFrame(index=index)
        for j, col in enumerate(self.columns_):
            vals = matrix[:, j]
            if col in self.numeric_columns_:
                out[col] = vals.astype(float)
            else:
                inv = self.inverse_category_maps_[col]
                max_code = max(inv.keys())
                codes = np.rint(vals).astype(int)
                codes = np.clip(codes, 0, max_code)
                out[col] = [inv[int(c)] for c in codes]
        return out[self.columns_]

# -----------------------------------------------------------------------------
# Weighted MLP classifier with class weighting
# -----------------------------------------------------------------------------

class WeightedMLPClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        learning_rate="adaptive",
        learning_rate_init=0.001,
        batch_size="auto",
        max_iter=2000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=40,
        tol=1e-4,
        class_weight_multiplier=2.0,
        majority_weight_divisor=2.0,
        resample_size_multiplier=1.5,
        random_state=42,
    ):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.activation = activation
        self.solver = solver
        self.alpha = alpha
        self.learning_rate = learning_rate
        self.learning_rate_init = learning_rate_init
        self.batch_size = batch_size
        self.max_iter = max_iter
        self.early_stopping = early_stopping
        self.validation_fraction = validation_fraction
        self.n_iter_no_change = n_iter_no_change
        self.tol = tol
        self.class_weight_multiplier = class_weight_multiplier
        self.majority_weight_divisor = majority_weight_divisor
        self.resample_size_multiplier = resample_size_multiplier
        self.random_state = random_state

    def fit(self, X, y):
        y = np.asarray(y)
        self.label_encoder_ = LabelEncoder()
        y_encoded = self.label_encoder_.fit_transform(y)
        self.classes_ = self.label_encoder_.classes_
        encoded_classes = np.arange(len(self.classes_))

        balanced_weights = compute_class_weight(class_weight="balanced", classes=encoded_classes, y=y_encoded)
        severe_weights = []
        for w in balanced_weights:
            if w >= 1.0:
                severe_weights.append(w * float(self.class_weight_multiplier))
            else:
                severe_weights.append(w / float(self.majority_weight_divisor))

        self.class_weight_ = {cls: float(w) for cls, w in zip(self.classes_, severe_weights)}
        encoded_weight_map = {enc: float(w) for enc, w in zip(encoded_classes, severe_weights)}
        sample_weight = np.array([encoded_weight_map[label] for label in y_encoded], dtype=float)

        self.mlp_ = MLPClassifier(
            hidden_layer_sizes=self.hidden_layer_sizes,
            activation=self.activation,
            solver=self.solver,
            alpha=self.alpha,
            learning_rate=self.learning_rate,
            learning_rate_init=self.learning_rate_init,
            batch_size=self.batch_size,
            max_iter=self.max_iter,
            early_stopping=self.early_stopping,
            validation_fraction=self.validation_fraction,
            n_iter_no_change=self.n_iter_no_change,
            tol=self.tol,
            random_state=self.random_state,
            shuffle=True,
        )

        fit_signature = inspect.signature(self.mlp_.fit)
        if "sample_weight" in fit_signature.parameters:
            self.weighting_method_ = "sample_weight"
            self.mlp_.fit(X, y_encoded, sample_weight=sample_weight)
        else:
            self.weighting_method_ = "weighted_resampling_fallback"
            rng = check_random_state(self.random_state)
            probabilities = sample_weight / sample_weight.sum()
            n = len(y_encoded)
            resample_n = int(max(n, round(n * float(self.resample_size_multiplier))))
            sampled_idx = rng.choice(np.arange(n), size=resample_n, replace=True, p=probabilities)
            forced_idx = []
            for cls_code in encoded_classes:
                cls_idx = np.where(y_encoded == cls_code)[0]
                if len(cls_idx) > 0:
                    forced_idx.append(rng.choice(cls_idx))
            sampled_idx = np.concatenate([sampled_idx, np.array(forced_idx, dtype=int)])
            self.mlp_.fit(X[sampled_idx], y_encoded[sampled_idx])

        self.n_features_in_ = self.mlp_.n_features_in_
        return self

    def predict(self, X):
        pred_encoded = self.mlp_.predict(X)
        return self.label_encoder_.inverse_transform(pred_encoded.astype(int))

    def predict_proba(self, X):
        return self.mlp_.predict_proba(X)

# -----------------------------------------------------------------------------
# Metrics and threshold tuning
# -----------------------------------------------------------------------------

def multiclass_specificity_score(y_true, y_pred, labels=None) -> float:
    labels = np.array(labels if labels is not None else np.unique(y_true))
    specificities = []
    for cls in labels:
        y_true_bin = np.asarray(y_true) == cls
        y_pred_bin = np.asarray(y_pred) == cls
        tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[False, True]).ravel()
        denom = tn + fp
        if denom > 0:
            specificities.append(tn / denom)
    return float(np.nanmean(specificities)) if specificities else np.nan


def align_proba(proba_raw, raw_classes, target_classes):
    aligned = np.zeros((proba_raw.shape[0], len(target_classes)), dtype=float)
    raw_classes = list(raw_classes)
    for j, cls in enumerate(target_classes):
        if cls in raw_classes:
            aligned[:, j] = proba_raw[:, raw_classes.index(cls)]
    return aligned


def calculate_metrics(y_true, y_pred, y_proba, classes) -> Dict[str, float]:
    results = {}
    try:
        results["ROC-AUC macro OvR"] = roc_auc_score(y_true, y_proba, labels=list(classes), multi_class="ovr", average="macro")
    except Exception:
        results["ROC-AUC macro OvR"] = np.nan

    y_bin = label_binarize(y_true, classes=list(classes))
    if len(classes) == 2 and y_bin.shape[1] == 1:
        y_bin = np.column_stack([1 - y_bin, y_bin])
    try:
        results["PR-AUC macro"] = average_precision_score(y_bin, y_proba, average="macro")
    except Exception:
        results["PR-AUC macro"] = np.nan

    results["F1 macro"] = f1_score(y_true, y_pred, average="macro", zero_division=0)
    results["F1 weighted"] = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    results["Accuracy"] = accuracy_score(y_true, y_pred)
    results["Precision macro"] = precision_score(y_true, y_pred, average="macro", zero_division=0)
    results["Recall macro"] = recall_score(y_true, y_pred, average="macro", zero_division=0)
    results["Specificity macro"] = multiclass_specificity_score(y_true, y_pred, labels=classes)
    return results


def composite_cv_score(estimator, X_val, y_val):
    try:
        proba = estimator.predict_proba(X_val)
        pred = estimator.predict(X_val)
        classes = list(estimator.named_steps["clf"].classes_)
        f1 = f1_score(y_val, pred, average="macro", zero_division=0)
        acc = accuracy_score(y_val, pred)
        try:
            roc = roc_auc_score(y_val, proba, labels=classes, multi_class="ovr", average="macro")
        except Exception:
            roc = 0.0
        return 0.50 * f1 + 0.30 * roc + 0.20 * acc
    except Exception:
        return 0.0


def predict_with_thresholds(y_proba, classes, thresholds, rule="ratio"):
    thresholds = np.asarray(thresholds, dtype=float)
    thresholds = np.clip(thresholds, 1e-6, 0.999999)
    if rule == "subtract":
        scores = y_proba - thresholds.reshape(1, -1)
    else:
        scores = y_proba / thresholds.reshape(1, -1)
    idx = np.argmax(scores, axis=1)
    return np.asarray(classes)[idx]


def threshold_objective_score(y_true, pred, objective="macro_f1_accuracy"):
    f1 = f1_score(y_true, pred, average="macro", zero_division=0)
    acc = accuracy_score(y_true, pred)
    rec = recall_score(y_true, pred, average="macro", zero_division=0)
    if objective == "macro_f1":
        return f1
    if objective == "balanced":
        return 0.65 * f1 + 0.20 * rec + 0.15 * acc
    return 0.75 * f1 + 0.25 * acc


def one_vs_rest_best_f1_thresholds(y_true, y_proba, classes):
    thresholds = []
    rows = []
    for i, cls in enumerate(classes):
        y_bin = (np.asarray(y_true) == cls).astype(int)
        p = y_proba[:, i]
        if y_bin.sum() == 0 or y_bin.sum() == len(y_bin):
            thresholds.append(0.5)
            rows.append({"Class": cls, "Prevalence": float(y_bin.mean()), "Best OVR F1 Threshold": 0.5, "Best OVR F1": np.nan})
            continue
        precision, recall, pr_thresholds = precision_recall_curve(y_bin, p)
        f1_vals = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
        if len(pr_thresholds) > 0:
            best_idx = int(np.nanargmax(f1_vals[:-1]))
            thr = float(pr_thresholds[best_idx])
            best_f1 = float(f1_vals[best_idx])
        else:
            thr = 0.5
            best_f1 = np.nan
        thresholds.append(np.clip(thr, 0.001, 0.999))
        rows.append({"Class": cls, "Prevalence": float(y_bin.mean()), "Best OVR F1 Threshold": thr, "Best OVR F1": best_f1})
    return np.array(thresholds, dtype=float), pd.DataFrame(rows)


def tune_multiclass_thresholds(y_true, y_proba, classes, n_iter=60000, random_state=42, rule="ratio", objective="macro_f1_accuracy"):
    rng = np.random.default_rng(random_state)
    classes = np.asarray(classes)
    y_true = np.asarray(y_true)
    n_classes = len(classes)

    def score(thr):
        pred = predict_with_thresholds(y_proba, classes, thr, rule=rule)
        return threshold_objective_score(y_true, pred, objective=objective)

    candidates = []
    candidates.append(np.ones(n_classes))
    candidates.append(np.full(n_classes, 0.5))
    ovr_thr, _ = one_vs_rest_best_f1_thresholds(y_true, y_proba, classes)
    candidates.append(ovr_thr)
    prevalence = np.array([(y_true == cls).mean() for cls in classes])
    candidates.append(np.clip(prevalence / np.maximum(prevalence.mean(), 1e-8), 0.001, 0.999))
    candidates.append(np.clip(np.sqrt(prevalence / np.maximum(prevalence.mean(), 1e-8)), 0.001, 0.999))

    best_thr = candidates[0].copy()
    best_score = score(best_thr)
    for cand in candidates[1:]:
        s = score(cand)
        if s > best_score:
            best_score = s
            best_thr = cand.copy()

    for _ in range(int(n_iter)):
        r = rng.random()
        if r < 0.50:
            noise = rng.normal(loc=0.0, scale=0.55, size=n_classes)
            cand = np.clip(best_thr * np.exp(noise), 0.001, 0.999)
        elif r < 0.73:
            cand = np.exp(rng.uniform(np.log(0.001), np.log(0.999), size=n_classes))
        elif r < 0.88:
            cand = rng.beta(a=0.6, b=0.6, size=n_classes)
            cand = np.clip(cand, 0.001, 0.999)
        else:
            cand = rng.uniform(0.001, 0.999, size=n_classes)
        s = score(cand)
        if s > best_score:
            best_score = s
            best_thr = cand.copy()

    grid_values = np.unique(np.r_[
        [0.001, 0.002, 0.005, 0.008],
        np.linspace(0.01, 0.10, 19),
        np.linspace(0.12, 0.50, 20),
        np.linspace(0.55, 0.95, 17),
        [0.975, 0.99, 0.995, 0.999],
    ])
    improved = True
    passes = 0
    while improved and passes < 6:
        improved = False
        passes += 1
        for k in range(n_classes):
            current_best_val = best_thr[k]
            for val in grid_values:
                cand = best_thr.copy()
                cand[k] = val
                s = score(cand)
                if s > best_score:
                    best_score = s
                    current_best_val = val
                    improved = True
            best_thr[k] = current_best_val

    final_pred = predict_with_thresholds(y_proba, classes, best_thr, rule=rule)
    final_macro_f1 = f1_score(y_true, final_pred, average="macro", zero_division=0)
    final_acc = accuracy_score(y_true, final_pred)
    final_obj = threshold_objective_score(y_true, final_pred, objective=objective)
    return best_thr, float(final_macro_f1), float(final_acc), float(final_obj), final_pred


def threshold_diagnostics(y_true, y_proba, classes, final_thresholds, final_rule):
    rows = []
    ovr_thr, ovr_df = one_vs_rest_best_f1_thresholds(y_true, y_proba, classes)
    for i, cls in enumerate(classes):
        y_bin = (np.asarray(y_true) == cls).astype(int)
        p = y_proba[:, i]
        if y_bin.sum() == 0 or y_bin.sum() == len(y_bin):
            youden_thr = np.nan
            best_ovr_thr = np.nan
            best_ovr_f1 = np.nan
        else:
            fpr, tpr, roc_thresholds = roc_curve(y_bin, p)
            youden = tpr - fpr
            youden_thr = float(roc_thresholds[int(np.argmax(youden))])
            best_ovr_thr = float(ovr_df.loc[ovr_df["Class"] == cls, "Best OVR F1 Threshold"].iloc[0])
            best_ovr_f1 = float(ovr_df.loc[ovr_df["Class"] == cls, "Best OVR F1"].iloc[0])

        rows.append({
            "Class": cls,
            "Prevalence in threshold tuning data": float(y_bin.mean()),
            "Youden Threshold": youden_thr,
            "Best One-vs-Rest F1 Threshold": best_ovr_thr,
            "Best One-vs-Rest F1": best_ovr_f1,
            "Final Tuned Multiclass Threshold": float(final_thresholds[i]),
            "Final Multiclass Rule": final_rule,
        })
    return pd.DataFrame(rows)


def stratified_bootstrap_auc_ci(y_true, y_proba, classes, n_bootstrap=1000, random_state=42):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    classes = np.asarray(classes)
    indices_by_class = {cls: np.where(y_true == cls)[0] for cls in classes}
    auc_values = []

    for _ in range(n_bootstrap):
        sampled_indices = []
        for cls in classes:
            idx = indices_by_class[cls]
            if len(idx) == 0:
                continue
            sampled_indices.extend(rng.choice(idx, size=len(idx), replace=True).tolist())
        sampled_indices = np.array(sampled_indices, dtype=int)
        try:
            auc_val = roc_auc_score(y_true[sampled_indices], y_proba[sampled_indices, :], labels=list(classes), multi_class="ovr", average="macro")
            if np.isfinite(auc_val):
                auc_values.append(auc_val)
        except Exception:
            continue

    if len(auc_values) < 20:
        return np.nan, np.nan, len(auc_values)
    lower, upper = np.percentile(auc_values, [2.5, 97.5])
    return float(lower), float(upper), len(auc_values)

# -----------------------------------------------------------------------------
# Plotting: ROC, PR, Decision Curve
# -----------------------------------------------------------------------------

def safe_filename(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(s)).strip("_").lower()


def plot_roc_by_class(y_true, y_proba, classes, output_dir):
    plt.figure(figsize=(8, 6))
    paths = []
    for i, cls in enumerate(classes):
        y_bin = (np.asarray(y_true) == cls).astype(int)
        fpr, tpr, _ = roc_curve(y_bin, y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{cls} AUC={roc_auc:.3f}")

        plt.figure(figsize=(7, 5))
        plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve - {cls}")
        plt.legend(loc="lower right")
        plt.tight_layout()
        pth = os.path.join(output_dir, f"roc_curve_{safe_filename(cls)}.png")
        plt.savefig(pth, dpi=300)
        plt.close()
        paths.append(pth)

    plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves by Class")
    plt.legend(loc="lower right")
    plt.tight_layout()
    path = os.path.join(output_dir, "roc_curve_by_class.png")
    plt.savefig(path, dpi=300)
    plt.close()
    return path, paths


# -----------------------------------------------------------------------------
# Calibration metrics/plots
# -----------------------------------------------------------------------------

def expected_calibration_error(y_true_binary, prob, n_bins=10):
    y_true_binary = np.asarray(y_true_binary).astype(int)
    prob = np.asarray(prob).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    rows = []
    for b in range(n_bins):
        left, right = bins[b], bins[b + 1]
        if b == n_bins - 1:
            mask = (prob >= left) & (prob <= right)
        else:
            mask = (prob >= left) & (prob < right)
        n = int(mask.sum())
        if n == 0:
            rows.append({"Bin": b + 1, "Bin Lower": left, "Bin Upper": right, "N": 0, "Mean Predicted Probability": np.nan, "Observed Fraction": np.nan, "Abs Gap": np.nan})
            continue
        conf = float(prob[mask].mean())
        acc = float(y_true_binary[mask].mean())
        gap = abs(acc - conf)
        ece += (n / len(prob)) * gap
        rows.append({"Bin": b + 1, "Bin Lower": left, "Bin Upper": right, "N": n, "Mean Predicted Probability": conf, "Observed Fraction": acc, "Abs Gap": gap})
    return float(ece), pd.DataFrame(rows)


def _softmax_matrix(z: np.ndarray) -> np.ndarray:
    """Numerically stable row-wise softmax."""
    z = np.asarray(z, dtype=float)
    z = z - np.nanmax(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.maximum(exp_z.sum(axis=1, keepdims=True), 1e-12)


def _logits_from_probabilities(probabilities: np.ndarray) -> np.ndarray:
    """Convert probabilities to stabilized log-probability logits."""
    p = np.asarray(probabilities, dtype=float)
    p = np.clip(p, 1e-8, 1.0)
    p = p / np.maximum(p.sum(axis=1, keepdims=True), 1e-12)
    return np.log(p)


def _class_codes(y_true, classes) -> np.ndarray:
    class_to_index = {cls: i for i, cls in enumerate(classes)}
    return np.array([class_to_index[v] for v in np.asarray(y_true)], dtype=int)


def _multiclass_nll(y_true, probabilities: np.ndarray, classes) -> float:
    """Mean multiclass negative log-likelihood."""
    codes = _class_codes(y_true, classes)
    p = np.asarray(probabilities, dtype=float)
    p = np.clip(p, 1e-8, 1.0)
    return float(-np.mean(np.log(p[np.arange(len(codes)), codes])))


def _multiclass_brier(y_true, probabilities: np.ndarray, classes) -> float:
    """Mean one-vs-rest Brier score across classes."""
    scores = []
    y_arr = np.asarray(y_true)
    for i, cls in enumerate(classes):
        y_bin = (y_arr == cls).astype(int)
        scores.append(brier_score_loss(y_bin, probabilities[:, i]))
    return float(np.nanmean(scores))


def _multiclass_ece(y_true, probabilities: np.ndarray, classes, n_bins: int = 10) -> float:
    """Mean one-vs-rest expected calibration error across classes."""
    eces = []
    y_arr = np.asarray(y_true)
    for i, cls in enumerate(classes):
        y_bin = (y_arr == cls).astype(int)
        ece, _ = expected_calibration_error(y_bin, probabilities[:, i], n_bins=n_bins)
        eces.append(ece)
    return float(np.nanmean(eces))


def _calibration_summary(y_true, probabilities: np.ndarray, classes, n_bins: int = 10) -> Dict[str, float]:
    """Summarize multiclass calibration quality."""
    nll = _multiclass_nll(y_true, probabilities, classes)
    brier = _multiclass_brier(y_true, probabilities, classes)
    ece = _multiclass_ece(y_true, probabilities, classes, n_bins=n_bins)
    return {
        "NLL": nll,
        "Mean Brier": brier,
        "Mean ECE": ece,
        "Calibration Objective": nll + brier + ece,
    }


def apply_probability_calibration(probabilities: np.ndarray, calibrator: Dict[str, object]) -> np.ndarray:
    """Apply the selected post hoc probability-calibration transform."""
    method = str(calibrator.get("method", "identity"))
    p = np.asarray(probabilities, dtype=float)
    p = np.clip(p, 1e-8, 1.0)
    p = p / np.maximum(p.sum(axis=1, keepdims=True), 1e-12)

    if method in {"identity", "none"}:
        return p

    logits = _logits_from_probabilities(p)

    if method == "temperature":
        temperature = float(calibrator.get("temperature", 1.0))
        temperature = max(temperature, 1e-4)
        return _softmax_matrix(logits / temperature)

    if method == "vector_scaling":
        scale = np.asarray(calibrator.get("scale"), dtype=float).reshape(1, -1)
        bias = np.asarray(calibrator.get("bias"), dtype=float).reshape(1, -1)
        return _softmax_matrix(logits * scale + bias)

    return p


def fit_probability_calibrator(
    y_true,
    probabilities: np.ndarray,
    classes,
    method: str = "auto",
    n_bins: int = 10,
    l2_penalty: float = 0.02,
    random_state: int = 42,
) -> Tuple[Dict[str, object], pd.DataFrame]:
    """
    Fit post hoc probability calibration using training-set out-of-fold predictions only.

    Candidate methods:
        identity        no calibration
        temperature     single multiclass temperature scaling
        vector_scaling  class-specific logit scaling and bias with L2 shrinkage

    In auto mode, the method with the lowest out-of-fold calibration objective
    is selected. The independent hold-out test set is not used to fit this
    calibration transform.
    """
    try:
        from scipy.optimize import minimize
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])
        from scipy.optimize import minimize

    y_arr = np.asarray(y_true)
    classes = np.asarray(classes)
    p = np.asarray(probabilities, dtype=float)
    p = np.clip(p, 1e-8, 1.0)
    p = p / np.maximum(p.sum(axis=1, keepdims=True), 1e-12)
    logits = _logits_from_probabilities(p)
    codes = _class_codes(y_arr, classes)
    n_classes = len(classes)

    requested_method = str(method).lower()
    if requested_method in {"none", "identity", "off"}:
        base_summary = _calibration_summary(y_arr, p, classes, n_bins=n_bins)
        calibrator = {"method": "identity", "classes": list(classes)}
        rows = [{"Candidate Method": "identity", "Selected": "Yes", **base_summary}]
        return calibrator, pd.DataFrame(rows)

    candidate_rows = []
    candidate_calibrators = []

    base_summary = _calibration_summary(y_arr, p, classes, n_bins=n_bins)
    candidate_calibrators.append({"method": "identity", "classes": list(classes)})
    candidate_rows.append({"Candidate Method": "identity", "Selected": "No", **base_summary})

    if requested_method in {"auto", "temperature"}:
        def temp_objective(theta):
            temperature = float(np.exp(theta[0]))
            cal = _softmax_matrix(logits / max(temperature, 1e-4))
            return _multiclass_nll(y_arr, cal, classes) + 1e-4 * float(theta[0] ** 2)

        res = minimize(temp_objective, x0=np.array([0.0]), method="L-BFGS-B", bounds=[(-3.0, 3.0)])
        temperature = float(np.exp(res.x[0])) if getattr(res, "success", False) else 1.0
        cal_temp = _softmax_matrix(logits / max(temperature, 1e-4))
        summary_temp = _calibration_summary(y_arr, cal_temp, classes, n_bins=n_bins)
        candidate_calibrators.append({
            "method": "temperature",
            "classes": list(classes),
            "temperature": temperature,
        })
        candidate_rows.append({
            "Candidate Method": "temperature",
            "Selected": "No",
            "Temperature": temperature,
            **summary_temp,
        })

    if requested_method in {"auto", "vector_scaling"}:
        rng = np.random.default_rng(random_state)
        x0 = np.zeros(2 * n_classes, dtype=float)
        x0[:n_classes] = 0.0
        x0[n_classes:] = rng.normal(0, 0.01, size=n_classes)

        def vector_objective(theta):
            log_scale = theta[:n_classes]
            bias = theta[n_classes:]
            scale = np.exp(log_scale).reshape(1, -1)
            bias_centered = (bias - np.mean(bias)).reshape(1, -1)
            cal = _softmax_matrix(logits * scale + bias_centered)
            nll = -np.mean(np.log(np.clip(cal[np.arange(len(codes)), codes], 1e-8, 1.0)))
            penalty = float(l2_penalty) * (np.sum(log_scale ** 2) + np.sum(bias_centered.ravel() ** 2))
            return float(nll + penalty)

        bounds = [(-2.5, 2.5)] * n_classes + [(-4.0, 4.0)] * n_classes
        res = minimize(vector_objective, x0=x0, method="L-BFGS-B", bounds=bounds, options={"maxiter": 500})
        theta = res.x if getattr(res, "success", False) else x0
        scale = np.exp(theta[:n_classes])
        bias = theta[n_classes:]
        bias = bias - np.mean(bias)
        cal_vec = _softmax_matrix(logits * scale.reshape(1, -1) + bias.reshape(1, -1))
        summary_vec = _calibration_summary(y_arr, cal_vec, classes, n_bins=n_bins)
        candidate_calibrators.append({
            "method": "vector_scaling",
            "classes": list(classes),
            "scale": scale.tolist(),
            "bias": bias.tolist(),
            "l2_penalty": float(l2_penalty),
        })
        candidate_rows.append({
            "Candidate Method": "vector_scaling",
            "Selected": "No",
            "L2 Penalty": float(l2_penalty),
            **summary_vec,
        })

    candidate_df = pd.DataFrame(candidate_rows)

    if requested_method == "temperature":
        selected_idx = candidate_df.index[candidate_df["Candidate Method"] == "temperature"][0]
    elif requested_method == "vector_scaling":
        selected_idx = candidate_df.index[candidate_df["Candidate Method"] == "vector_scaling"][0]
    else:
        selected_idx = pd.to_numeric(candidate_df["Calibration Objective"], errors="coerce").idxmin()

    candidate_df.loc[:, "Selected"] = "No"
    candidate_df.loc[selected_idx, "Selected"] = "Yes"

    selected_method = candidate_df.loc[selected_idx, "Candidate Method"]
    calibrator = candidate_calibrators[list(candidate_df["Candidate Method"]).index(selected_method)]
    calibrator["oof_selection_metric"] = "lowest NLL + mean Brier + mean ECE on training-set OOF predictions"
    calibrator["oof_before"] = base_summary
    calibrator["oof_selected"] = {
        k: float(candidate_df.loc[selected_idx, k])
        for k in ["NLL", "Mean Brier", "Mean ECE", "Calibration Objective"]
        if k in candidate_df.columns and pd.notna(candidate_df.loc[selected_idx, k])
    }
    return calibrator, candidate_df


def plot_calibration_curves(y_true, y_proba, classes, output_dir, n_bins=10):
    paths = []
    metric_rows = []
    bin_frames = []

    plt.figure(figsize=(8, 6))
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

    for i, cls in enumerate(classes):
        y_bin = (np.asarray(y_true) == cls).astype(int)
        p = y_proba[:, i]
        brier = brier_score_loss(y_bin, p)
        ece, bin_df = expected_calibration_error(y_bin, p, n_bins=n_bins)
        bin_df.insert(0, "Class", cls)
        bin_frames.append(bin_df)
        try:
            frac_pos, mean_pred = calibration_curve(y_bin, p, n_bins=n_bins, strategy="quantile")
        except Exception:
            frac_pos, mean_pred = calibration_curve(y_bin, p, n_bins=n_bins, strategy="uniform")

        metric_rows.append({"Class": cls, "Brier Score": brier, "ECE": ece, "Calibration Bins": n_bins})

        plt.plot(mean_pred, frac_pos, marker="o", label=f"{cls} (Brier={brier:.3f}, ECE={ece:.3f})")

        plt.figure(figsize=(7, 5))
        plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
        plt.plot(mean_pred, frac_pos, marker="o", label=f"Model")
        plt.xlabel("Mean Predicted Probability")
        plt.ylabel("Observed Fraction")
        plt.title(f"Calibration Curve - {cls}\nBrier={brier:.4f}, ECE={ece:.4f}")
        plt.legend(loc="best")
        plt.tight_layout()
        pth = os.path.join(output_dir, f"calibration_curve_{safe_filename(cls)}.png")
        plt.savefig(pth, dpi=300)
        plt.close()
        paths.append(pth)

    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Observed Fraction")
    plt.title("Calibration Curves by Class")
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    combined_path = os.path.join(output_dir, "calibration_curve_by_class.png")
    plt.savefig(combined_path, dpi=300)
    plt.close()

    calibration_metrics_df = pd.DataFrame(metric_rows)
    calibration_bins_df = pd.concat(bin_frames, ignore_index=True) if bin_frames else pd.DataFrame()
    return combined_path, paths, calibration_metrics_df, calibration_bins_df

# -----------------------------------------------------------------------------
# Learning curve
# -----------------------------------------------------------------------------

def plot_learning_curve(estimator, X, y, cv, output_dir, n_jobs=-1):
    train_sizes = np.linspace(0.20, 1.00, 5)
    train_sizes_abs, train_scores, valid_scores = learning_curve(
        estimator=estimator,
        X=X,
        y=y,
        train_sizes=train_sizes,
        cv=cv,
        scoring="f1_macro",
        n_jobs=n_jobs,
        shuffle=True,
        random_state=42,
    )

    rows = []
    for i, n in enumerate(train_sizes_abs):
        rows.append({
            "Training Size": int(n),
            "Train Macro-F1 Mean": float(np.nanmean(train_scores[i])),
            "Train Macro-F1 SD": float(np.nanstd(train_scores[i], ddof=1)),
            "Validation Macro-F1 Mean": float(np.nanmean(valid_scores[i])),
            "Validation Macro-F1 SD": float(np.nanstd(valid_scores[i], ddof=1)),
        })
    df = pd.DataFrame(rows)

    plt.figure(figsize=(8, 6))
    plt.plot(df["Training Size"], df["Train Macro-F1 Mean"], marker="o", label="Training macro-F1")
    plt.fill_between(
        df["Training Size"],
        df["Train Macro-F1 Mean"] - df["Train Macro-F1 SD"],
        df["Train Macro-F1 Mean"] + df["Train Macro-F1 SD"],
        alpha=0.15,
    )
    plt.plot(df["Training Size"], df["Validation Macro-F1 Mean"], marker="o", label="CV validation macro-F1")
    plt.fill_between(
        df["Training Size"],
        df["Validation Macro-F1 Mean"] - df["Validation Macro-F1 SD"],
        df["Validation Macro-F1 Mean"] + df["Validation Macro-F1 SD"],
        alpha=0.15,
    )
    plt.xlabel("Training Set Size")
    plt.ylabel("Macro-F1")
    plt.title("Learning Curve - Macro-F1")
    plt.legend(loc="best")
    plt.tight_layout()
    path = os.path.join(output_dir, "learning_curve_macro_f1.png")
    plt.savefig(path, dpi=300)
    plt.close()
    return path, df

# -----------------------------------------------------------------------------
# SHAP
# -----------------------------------------------------------------------------

def install_import_shap():
    try:
        import shap  # noqa
        return shap
    except Exception:
        print("Installing shap...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
        import shap  # noqa
        return shap


def get_preprocessed_feature_names(pipeline):
    preprocess = pipeline.named_steps["preprocess"]
    try:
        names = preprocess.get_feature_names_out()
        return [str(n) for n in names]
    except Exception:
        names = []
        names.extend(NUMERIC_FEATURES)
        names.extend(ORDINAL_FEATURES)
        names.extend(BINARY_FEATURES)
        for col in NOMINAL_FEATURES:
            cats = KNOWN_NOMINAL_CATEGORIES.get(col, [])
            for cat in cats:
                names.append(f"{col}_{cat}")
        return names


def shap_values_to_array(shap_values):
    # shap.Explanation values usually shape = (n_samples, n_features, n_classes)
    if hasattr(shap_values, "values"):
        return shap_values.values, shap_values.base_values
    if isinstance(shap_values, list):
        arr = np.stack(shap_values, axis=2)
        return arr, None
    return np.asarray(shap_values), None




SHAP_LOW_COLOR = "#1E88E5"
SHAP_HIGH_COLOR = "#FF0052"


def original_feature_from_preprocessed_name(feature_name: str) -> str:
    """Map a transformed feature name back to the prespecified original feature name."""
    name = str(feature_name)
    if name in ALL_FEATURES:
        return name
    for prefix in ["numeric__", "ordinal__", "binary__", "nominal__"]:
        if name.startswith(prefix):
            name = name[len(prefix):]
            break
    if name in ALL_FEATURES:
        return name
    for feature in NOMINAL_FEATURES:
        if name.startswith(feature + "_"):
            return feature
    for feature in ALL_FEATURES:
        if name.startswith(feature + "_"):
            return feature
    return name


def encode_original_feature_for_shap_display(series: pd.Series, feature: str) -> np.ndarray:
    """Convert original feature values to numeric display values for grouped SHAP beeswarm coloring."""
    if feature in NUMERIC_FEATURES:
        return pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    if feature in ORDINAL_FEATURES:
        cats = ORDINAL_CATEGORIES.get(feature, [])
        mapping = {cat: i for i, cat in enumerate(cats)}
        return series.map(mapping).astype(float).to_numpy()
    if feature in BINARY_FEATURES:
        cats = BINARY_CATEGORIES.get(feature, [])
        mapping = {cat: i for i, cat in enumerate(cats)}
        return series.map(mapping).astype(float).to_numpy()
    if feature in NOMINAL_FEATURES:
        cats = KNOWN_NOMINAL_CATEGORIES.get(feature, [])
        mapping = {cat: i for i, cat in enumerate(cats)}
        return series.map(mapping).astype(float).to_numpy()
    return pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)


def build_grouped_shap_explanation(
    values: np.ndarray,
    base_values,
    class_idx: int,
    feature_names: List[str],
    X_original_level: pd.DataFrame,
):
    """Aggregate one-hot/preprocessed SHAP columns back to one row per original feature."""
    grouped_feature_names = [f for f in ALL_FEATURES if f in X_original_level.columns]
    if not grouped_feature_names:
        grouped_feature_names = list(dict.fromkeys(original_feature_from_preprocessed_name(f) for f in feature_names))

    grouped_values = np.zeros((values.shape[0], len(grouped_feature_names)), dtype=float)
    feature_to_group_index = {feature: i for i, feature in enumerate(grouped_feature_names)}

    for j, transformed_name in enumerate(feature_names):
        original_name = original_feature_from_preprocessed_name(transformed_name)
        if original_name in feature_to_group_index:
            grouped_values[:, feature_to_group_index[original_name]] += values[:, j, class_idx]

    grouped_display = np.zeros((values.shape[0], len(grouped_feature_names)), dtype=float)
    for j, feature in enumerate(grouped_feature_names):
        if feature in X_original_level.columns:
            grouped_display[:, j] = encode_original_feature_for_shap_display(X_original_level[feature], feature)
        else:
            grouped_display[:, j] = np.nan

    if base_values is not None and np.ndim(base_values) == 2:
        grouped_base = base_values[:, class_idx]
    else:
        grouped_base = None

    shap = install_import_shap()
    grouped_exp = shap.Explanation(
        values=grouped_values,
        base_values=grouped_base,
        data=grouped_display,
        feature_names=grouped_feature_names,
    )
    return grouped_exp, grouped_values, grouped_display, grouped_feature_names

def _age_threshold_from_shap(age_values: np.ndarray, shap_values: np.ndarray) -> float:
    """Identify an age threshold that maximizes separation in age-specific SHAP contribution."""
    age_values = np.asarray(age_values, dtype=float)
    shap_values = np.asarray(shap_values, dtype=float)
    valid = np.isfinite(age_values) & np.isfinite(shap_values)
    age_values = age_values[valid]
    shap_values = shap_values[valid]
    if len(age_values) < 20 or len(np.unique(age_values)) < 5:
        return float(np.nanmedian(age_values)) if len(age_values) else np.nan

    candidates = np.unique(np.percentile(age_values, np.linspace(10, 90, 81)))
    min_group = max(5, int(0.10 * len(age_values)))
    best_threshold = float(np.nanmedian(age_values))
    best_score = -np.inf
    for threshold in candidates:
        lower = age_values <= threshold
        upper = age_values > threshold
        if lower.sum() < min_group or upper.sum() < min_group:
            continue
        separation = abs(float(np.nanmean(shap_values[upper])) - float(np.nanmean(shap_values[lower])))
        balance = min(lower.sum(), upper.sum()) / len(age_values)
        score = separation * balance
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)
    return best_threshold


def _rolling_median_line(x_values: np.ndarray, y_values: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Create a stable rolling median curve for SHAP threshold visualization."""
    x_values = np.asarray(x_values, dtype=float)
    y_values = np.asarray(y_values, dtype=float)
    valid = np.isfinite(x_values) & np.isfinite(y_values)
    x_values = x_values[valid]
    y_values = y_values[valid]
    if len(x_values) < 5:
        return x_values, y_values
    order = np.argsort(x_values)
    x_sorted = x_values[order]
    y_sorted = y_values[order]
    window = max(5, int(round(len(x_sorted) * 0.12)))
    if window % 2 == 0:
        window += 1
    y_med = pd.Series(y_sorted).rolling(window=window, center=True, min_periods=max(3, window // 3)).median().to_numpy()
    return x_sorted, y_med


def _plot_age_shap_threshold(
    age_values: np.ndarray,
    age_shap_values: np.ndarray,
    y_explain: pd.Series,
    predicted_probabilities: np.ndarray,
    class_index: int,
    class_name: str,
    output_dir: str,
) -> Tuple[str, Dict[str, object]]:
    """Plot class-specific age SHAP threshold and quantify observed class-rate change."""
    from matplotlib.colors import LinearSegmentedColormap

    age_values = np.asarray(age_values, dtype=float)
    age_shap_values = np.asarray(age_shap_values, dtype=float)
    valid = np.isfinite(age_values) & np.isfinite(age_shap_values)
    age = age_values[valid]
    shap_age = age_shap_values[valid]
    y_valid = pd.Series(y_explain).reset_index(drop=True).iloc[np.where(valid)[0]].to_numpy()
    prob_valid = np.asarray(predicted_probabilities, dtype=float)[valid, class_index]

    threshold = _age_threshold_from_shap(age, shap_age)
    above = age > threshold
    below = age <= threshold

    observed_rate_above = float(np.mean(y_valid[above] == class_name) * 100) if above.sum() else np.nan
    observed_rate_below = float(np.mean(y_valid[below] == class_name) * 100) if below.sum() else np.nan
    absolute_increase = observed_rate_above - observed_rate_below if np.isfinite(observed_rate_above) and np.isfinite(observed_rate_below) else np.nan
    relative_increase = ((observed_rate_above / observed_rate_below) - 1.0) * 100 if np.isfinite(observed_rate_below) and observed_rate_below > 0 else np.nan
    mean_pred_above = float(np.mean(prob_valid[above]) * 100) if above.sum() else np.nan
    mean_pred_below = float(np.mean(prob_valid[below]) * 100) if below.sum() else np.nan
    mean_pred_absolute_increase = mean_pred_above - mean_pred_below if np.isfinite(mean_pred_above) and np.isfinite(mean_pred_below) else np.nan

    cmap = LinearSegmentedColormap.from_list("shap_blue_pink", [SHAP_LOW_COLOR, SHAP_HIGH_COLOR])
    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    scatter = ax.scatter(age, shap_age, c=age, cmap=cmap, s=28, alpha=0.85, edgecolor="none")
    x_line, y_line = _rolling_median_line(age, shap_age)
    if len(x_line):
        ax.plot(x_line, y_line, linewidth=2.2, color="black", alpha=0.80, label="Rolling median")
    ax.axhline(0, color="gray", linestyle="--", linewidth=1.0)
    if np.isfinite(threshold):
        ax.axvline(threshold, color="black", linestyle="--", linewidth=1.3, label=f"Threshold = {threshold:.1f} years")

    annotation = (
        f"Age threshold: {threshold:.1f} years\n"
        f"Observed {class_name} rate > threshold: {observed_rate_above:.1f}%\n"
        f"Observed {class_name} rate ≤ threshold: {observed_rate_below:.1f}%\n"
        f"Absolute change: {absolute_increase:+.1f} percentage points\n"
        f"Relative change: {relative_increase:+.1f}%"
    )
    ax.text(
        0.02, 0.98, annotation,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="gray", alpha=0.92),
    )
    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label("Age (Years)")
    ax.set_xlabel("Age (Years)")
    ax.set_ylabel("SHAP value for Age")
    ax.set_title(f"Age SHAP Threshold - {class_name}")
    ax.legend(loc="lower right")
    plt.tight_layout()
    path = os.path.join(output_dir, f"shap_age_threshold_{safe_filename(class_name)}.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

    row = {
        "Class": class_name,
        "Age Threshold (Years)": threshold,
        "N <= Threshold": int(below.sum()),
        "N > Threshold": int(above.sum()),
        "Observed Class Rate <= Threshold (%)": observed_rate_below,
        "Observed Class Rate > Threshold (%)": observed_rate_above,
        "Absolute Increase Above Threshold (percentage points)": absolute_increase,
        "Relative Increase Above Threshold (%)": relative_increase,
        "Mean Predicted Probability <= Threshold (%)": mean_pred_below,
        "Mean Predicted Probability > Threshold (%)": mean_pred_above,
        "Mean Predicted Probability Absolute Increase (percentage points)": mean_pred_absolute_increase,
        "Plot Path": path,
    }
    return path, row


def run_shap_plots(
    best_model,
    X_train,
    X_test,
    y_test,
    classes,
    output_dir,
    background_size=100,
    explain_size=None,
    max_evals=800,
    random_state=42,
    probability_calibrator=None,
):
    """
    Perform post hoc SHAP analysis on the hold-out test set only.

    SHAP results are generated after final model selection and evaluation. They are
    used only for interpretation and do not influence model selection,
    hyperparameter tuning, threshold tuning, or performance estimation.

    One-hot encoded and otherwise transformed columns are aggregated back to the
    original feature level before beeswarm plotting, so each original predictor is
    represented by a single row in each class-specific SHAP beeswarm plot.
    """
    shap = install_import_shap()
    from matplotlib.colors import LinearSegmentedColormap

    shap_cmap = LinearSegmentedColormap.from_list("shap_blue_pink", [SHAP_LOW_COLOR, SHAP_HIGH_COLOR])

    bg_n = min(background_size, len(X_train))
    ex_n = len(X_test) if explain_size is None else min(explain_size, len(X_test))
    X_bg_raw = X_train.sample(n=bg_n, random_state=random_state).copy()
    X_ex_raw = X_test.sample(n=ex_n, random_state=random_state + 1).copy()
    y_explain = pd.Series(y_test).loc[X_ex_raw.index].copy()

    X_bg_mice = best_model.named_steps["mice"].transform(X_bg_raw)
    X_ex_mice = best_model.named_steps["mice"].transform(X_ex_raw)
    X_bg_pre = best_model.named_steps["preprocess"].transform(X_bg_mice)
    X_ex_pre = best_model.named_steps["preprocess"].transform(X_ex_mice)

    if hasattr(X_bg_pre, "toarray"):
        X_bg_pre = X_bg_pre.toarray()
    if hasattr(X_ex_pre, "toarray"):
        X_ex_pre = X_ex_pre.toarray()
    X_bg_pre = np.asarray(X_bg_pre, dtype=float)
    X_ex_pre = np.asarray(X_ex_pre, dtype=float)

    feature_names = get_preprocessed_feature_names(best_model)
    if len(feature_names) != X_ex_pre.shape[1]:
        feature_names = [f"Feature_{i}" for i in range(X_ex_pre.shape[1])]

    print(f"Computing post hoc SHAP values on {X_ex_pre.shape[0]} hold-out test samples and {X_ex_pre.shape[1]} preprocessed features...")
    masker = shap.maskers.Independent(X_bg_pre)
    explainer = shap.Explainer(
        best_model.named_steps["clf"].mlp_.predict_proba,
        masker,
        algorithm="permutation",
        feature_names=feature_names,
        max_evals=max_evals,
    )
    shap_values = explainer(X_ex_pre)
    values, base_values = shap_values_to_array(shap_values)

    if values.ndim != 3:
        raise RuntimeError(f"Unexpected SHAP values shape: {values.shape}; expected (samples, features, classes).")

    predicted_probabilities = align_proba(
        best_model.predict_proba(X_ex_raw),
        best_model.named_steps["clf"].classes_,
        classes,
    )
    if probability_calibrator is not None:
        predicted_probabilities = apply_probability_calibration(predicted_probabilities, probability_calibrator)

    paths = []
    importance_frames = []
    age_threshold_rows = []

    for class_idx, cls in enumerate(classes):
        class_safe = safe_filename(cls)
        grouped_exp, grouped_values, grouped_display, grouped_feature_names = build_grouped_shap_explanation(
            values=values,
            base_values=base_values,
            class_idx=class_idx,
            feature_names=feature_names,
            X_original_level=X_ex_mice,
        )

        plt.figure(figsize=(10, 7))
        shap.plots.beeswarm(
            grouped_exp,
            max_display=min(25, len(grouped_feature_names)),
            color=shap_cmap,
            show=False,
        )
        plt.title(f"Grouped SHAP Beeswarm - {cls}")
        plt.tight_layout()
        beeswarm_path = os.path.join(output_dir, f"shap_beeswarm_{class_safe}.png")
        plt.savefig(beeswarm_path, dpi=300, bbox_inches="tight")
        plt.close()
        paths.append(beeswarm_path)

        if "Age (Years)" in grouped_feature_names and "Age (Years)" in X_ex_mice.columns:
            age_group_idx = grouped_feature_names.index("Age (Years)")
            age_path, age_row = _plot_age_shap_threshold(
                age_values=pd.to_numeric(X_ex_mice["Age (Years)"], errors="coerce").to_numpy(),
                age_shap_values=grouped_values[:, age_group_idx],
                y_explain=y_explain,
                predicted_probabilities=predicted_probabilities,
                class_index=class_idx,
                class_name=str(cls),
                output_dir=output_dir,
            )
            paths.append(age_path)
            age_threshold_rows.append(age_row)

        mean_abs = np.abs(grouped_values).mean(axis=0)
        importance_frames.append(pd.DataFrame({
            "Class": cls,
            "Original Feature": grouped_feature_names,
            "Mean |SHAP|": mean_abs,
        }).sort_values("Mean |SHAP|", ascending=False))

    shap_importance_df = pd.concat(importance_frames, ignore_index=True) if importance_frames else pd.DataFrame()
    age_threshold_df = pd.DataFrame(age_threshold_rows)
    return paths, shap_importance_df, age_threshold_df

# -----------------------------------------------------------------------------
# Pipeline and search space
# -----------------------------------------------------------------------------

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_pipeline(random_state=42) -> Pipeline:
    known_categories = {}
    known_categories.update(ORDINAL_CATEGORIES)
    known_categories.update(BINARY_CATEGORIES)
    known_categories.update(KNOWN_NOMINAL_CATEGORIES)

    mice = MixedTypeMICEImputer(
        numeric_columns=NUMERIC_FEATURES,
        categorical_columns=ORDINAL_FEATURES + BINARY_FEATURES + NOMINAL_FEATURES,
        known_categories=known_categories,
        max_iter=30,
        random_state=random_state,
    )

    ordinal_encoder = OrdinalEncoder(
        categories=[ORDINAL_CATEGORIES[col] for col in ORDINAL_FEATURES],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )

    binary_encoder = OrdinalEncoder(
        categories=[BINARY_CATEGORIES[col] for col in BINARY_FEATURES],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )

    numeric_pipe = Pipeline([("scaler", StandardScaler())])
    ordinal_pipe = Pipeline([("encoder", ordinal_encoder), ("scaler", StandardScaler())])
    binary_pipe = Pipeline([("encoder", binary_encoder), ("scaler", StandardScaler())])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipe, NUMERIC_FEATURES),
            ("ordinal", ordinal_pipe, ORDINAL_FEATURES),
            ("binary", binary_pipe, BINARY_FEATURES),
            ("nominal", make_one_hot_encoder(), NOMINAL_FEATURES),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    clf = WeightedMLPClassifier(random_state=random_state)

    return Pipeline([
        ("mice", mice),
        ("preprocess", preprocessor),
        ("clf", clf),
    ])


def get_randomized_parameter_space():
    return {
        "preprocess__numeric__scaler": [StandardScaler(), RobustScaler()],
        "preprocess__ordinal__scaler": [StandardScaler(), RobustScaler()],
        "preprocess__binary__scaler": [StandardScaler(), RobustScaler()],
        "clf__hidden_layer_sizes": [
            (32,), (48,), (64,), (96,), (128,), (192,), (256,), (384,),
            (64, 32), (96, 48), (128, 64), (192, 96), (256, 128), (384, 192),
            (128, 64, 32), (192, 96, 48), (256, 128, 64), (384, 192, 96),
            (256, 128, 64, 32),
        ],
        "clf__activation": ["relu", "tanh", "logistic"],
        "clf__alpha": [1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1],
        "clf__learning_rate": ["constant", "adaptive", "invscaling"],
        "clf__learning_rate_init": [3e-5, 7e-5, 1e-4, 3e-4, 7e-4, 1e-3, 2e-3, 3e-3, 5e-3, 7e-3],
        "clf__batch_size": [8, 16, 32, 64, 128, "auto"],
        "clf__early_stopping": [True],
        "clf__validation_fraction": [0.10, 0.15, 0.20, 0.25, 0.30],
        "clf__n_iter_no_change": [25, 35, 50, 70],
        "clf__tol": [1e-4, 5e-5, 1e-5, 5e-6],
        "clf__max_iter": [1200, 1800, 2400, 3000],
        "clf__class_weight_multiplier": [1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0],
        "clf__majority_weight_divisor": [1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0],
        "clf__resample_size_multiplier": [1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0],
    }


def manual_cross_validated_auc(estimator, X, y, cv, classes) -> np.ndarray:
    scores = []
    X_reset = X.reset_index(drop=True) if hasattr(X, "reset_index") else X
    y_series = pd.Series(y).reset_index(drop=True)
    for train_idx, valid_idx in cv.split(X_reset, y_series):
        fold_model = clone(estimator)
        X_tr = X_reset.iloc[train_idx] if hasattr(X_reset, "iloc") else X_reset[train_idx]
        X_va = X_reset.iloc[valid_idx] if hasattr(X_reset, "iloc") else X_reset[valid_idx]
        y_tr = y_series.iloc[train_idx]
        y_va = y_series.iloc[valid_idx]
        fold_model.fit(X_tr, y_tr)
        fold_proba_raw = fold_model.predict_proba(X_va)
        fold_classes = list(fold_model.named_steps["clf"].classes_)
        aligned = align_proba(fold_proba_raw, fold_classes, classes)
        try:
            score = roc_auc_score(y_va, aligned, labels=list(classes), multi_class="ovr", average="macro")
        except Exception:
            score = np.nan
        scores.append(score)
    return np.array(scores, dtype=float)


def out_of_fold_predictions(estimator, X, y, cv, classes):
    X_reset = X.reset_index(drop=True)
    y_series = pd.Series(y).reset_index(drop=True)
    oof_proba = np.zeros((len(y_series), len(classes)), dtype=float)
    oof_pred = np.empty(len(y_series), dtype=object)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_reset, y_series), start=1):
        print(f"  OOF threshold-tuning fold {fold}/{cv.n_splits}...")
        model = clone(estimator)
        model.fit(X_reset.iloc[train_idx], y_series.iloc[train_idx])
        raw_proba = model.predict_proba(X_reset.iloc[valid_idx])
        raw_classes = list(model.named_steps["clf"].classes_)
        aligned = align_proba(raw_proba, raw_classes, classes)
        oof_proba[valid_idx, :] = aligned
        oof_pred[valid_idx] = np.asarray(classes)[np.argmax(aligned, axis=1)]

    return y_series.to_numpy(), oof_pred, oof_proba


# -----------------------------------------------------------------------------
# Final analysis configuration
# -----------------------------------------------------------------------------

SCENARIOS = [
    {
        "scenario_id": "final_macro_f1_model",
        "scenario_label": "Final prespecified model | 1.00*macro-F1",
        "include_anticoagulant": True,
        "refit_metric": "f1_macro",
        "metric_definition": "1.00*macro-F1",
    },
]


def scenario_sort_key(row):
    try:
        return int(str(row["Scenario ID"]).split("_")[0])
    except Exception:
        return 999


def run_single_scenario(args, df_raw: pd.DataFrame, scenario: Dict[str, object]) -> Dict[str, object]:
    scenario_id = str(scenario["scenario_id"])
    scenario_label = str(scenario["scenario_label"])
    scenario_dir = args.output_dir
    os.makedirs(scenario_dir, exist_ok=True)

    include_anticoagulant = bool(scenario["include_anticoagulant"])
    configure_feature_set(include_anticoagulant)

    if include_anticoagulant and find_column(df_raw, ANTICOAGULANT_ALIASES) is None:
        raise ValueError(
            "Anticoagulant column was not found. This final prespecified analysis requires the Anticoagulant predictor."
        )

    print("\n" + "=" * 80)
    print("Running final analysis")
    print(f"{scenario_label}")
    print("=" * 80)

    if find_column(df_raw, FEATURE_ALIASES["Troponin"]) is None:
        raise ValueError("Troponin column was not found in the input file.")

    data, target_col = standardize_dataframe(df_raw)
    X = data[ALL_FEATURES].copy()
    y = data[target_col].copy()

    class_counts = y.value_counts().reindex(TARGET_CLASSES).fillna(0).astype(int)
    print("Target class counts:")
    print(class_counts.to_string())
    print(f"Features used ({len(ALL_FEATURES)}): " + ", ".join(ALL_FEATURES))

    present_classes = [c for c in TARGET_CLASSES if c in set(y)]
    if len(present_classes) < 2:
        raise ValueError("At least two target classes are required for classification.")
    classes = np.array(present_classes)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=args.test_size,
        random_state=args.random_state,
        stratify=y,
    )

    min_train_class_count = int(y_train.value_counts().min())
    n_splits = min(5, min_train_class_count)
    if n_splits < 2:
        raise ValueError("Each modeled target class must have at least 2 training rows.")
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=args.random_state)

    pipeline = build_pipeline(random_state=args.random_state)
    parameter_space = get_randomized_parameter_space()

    scoring = {
        "composite": composite_cv_score,
        "f1_macro": "f1_macro",
        "roc_auc_ovr_macro": "roc_auc_ovr",
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "specificity_macro": make_scorer(multiclass_specificity_score),
    }

    print(f"CV design: Stratified {n_splits}-fold CV")
    print(f"Refit / selection metric: {scenario['metric_definition']}")
    print(f"Candidates sampled: {args.n_iter}")

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=parameter_space,
        n_iter=args.n_iter,
        scoring=scoring,
        refit=str(scenario["refit_metric"]),
        cv=cv,
        n_jobs=args.n_jobs,
        verbose=args.verbose,
        random_state=args.random_state,
        return_train_score=True,
    )
    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    clf = best_model.named_steps["clf"]
    model_classes = np.array(list(clf.classes_))

    print("Computing out-of-fold probabilities for threshold tuning and probability calibration...")
    y_oof, y_oof_argmax_raw, oof_proba_raw = out_of_fold_predictions(best_model, X_train, y_train, cv, model_classes)

    probability_calibrator, probability_calibration_df = fit_probability_calibrator(
        y_oof,
        oof_proba_raw,
        model_classes,
        method=args.probability_calibration,
        n_bins=args.calibration_bins,
        l2_penalty=args.probability_calibration_l2,
        random_state=args.random_state,
    )
    oof_proba = apply_probability_calibration(oof_proba_raw, probability_calibrator)
    y_oof_argmax = np.asarray(model_classes)[np.argmax(oof_proba, axis=1)]
    oof_argmax_f1 = f1_score(y_oof, y_oof_argmax, average="macro", zero_division=0)
    oof_argmax_acc = accuracy_score(y_oof, y_oof_argmax)
    oof_argmax_obj = threshold_objective_score(y_oof, y_oof_argmax, objective=args.threshold_objective)

    print(f"Selected probability calibration method: {probability_calibrator.get('method', 'identity')}")
    print("Threshold tuning uses calibrated OOF probabilities from the training set only.")

    thresholds_ratio, oof_thr_f1_ratio, oof_thr_acc_ratio, oof_obj_ratio, _ = tune_multiclass_thresholds(
        y_oof, oof_proba, model_classes,
        n_iter=args.threshold_iter,
        random_state=args.random_state,
        rule="ratio",
        objective=args.threshold_objective,
    )
    thresholds_subtract, oof_thr_f1_subtract, oof_thr_acc_subtract, oof_obj_subtract, _ = tune_multiclass_thresholds(
        y_oof, oof_proba, model_classes,
        n_iter=max(2000, args.threshold_iter // 2),
        random_state=args.random_state + 17,
        rule="subtract",
        objective=args.threshold_objective,
    )

    if oof_obj_ratio >= oof_obj_subtract:
        final_thresholds = thresholds_ratio
        final_threshold_rule = "ratio: predict argmax(probability / class_threshold)"
        final_threshold_rule_key = "ratio"
        oof_threshold_f1 = oof_thr_f1_ratio
        oof_threshold_acc = oof_thr_acc_ratio
        oof_threshold_obj = oof_obj_ratio
    else:
        final_thresholds = thresholds_subtract
        final_threshold_rule = "subtract: predict argmax(probability - class_threshold)"
        final_threshold_rule_key = "subtract"
        oof_threshold_f1 = oof_thr_f1_subtract
        oof_threshold_acc = oof_thr_acc_subtract
        oof_threshold_obj = oof_obj_subtract

    if oof_threshold_obj <= oof_argmax_obj:
        final_threshold_rule = "argmax predicted probability"
        final_threshold_rule_key = "argmax"
        final_thresholds = np.ones(len(model_classes))
        oof_threshold_f1 = oof_argmax_f1
        oof_threshold_acc = oof_argmax_acc
        oof_threshold_obj = oof_argmax_obj

    best_model.fit(X_train, y_train)
    clf = best_model.named_steps["clf"]
    model_classes = np.array(list(clf.classes_))

    y_train_proba_raw = align_proba(best_model.predict_proba(X_train), clf.classes_, model_classes)
    y_test_proba_raw = align_proba(best_model.predict_proba(X_test), clf.classes_, model_classes)
    y_train_proba = apply_probability_calibration(y_train_proba_raw, probability_calibrator)
    y_test_proba = apply_probability_calibration(y_test_proba_raw, probability_calibrator)

    y_train_argmax = np.asarray(model_classes)[np.argmax(y_train_proba, axis=1)]
    y_test_argmax = np.asarray(model_classes)[np.argmax(y_test_proba, axis=1)]

    if final_threshold_rule_key == "argmax":
        y_train_pred = y_train_argmax
        y_test_pred = y_test_argmax
    else:
        y_train_pred = predict_with_thresholds(y_train_proba, model_classes, final_thresholds, rule=final_threshold_rule_key)
        y_test_pred = predict_with_thresholds(y_test_proba, model_classes, final_thresholds, rule=final_threshold_rule_key)

    train_metrics = calculate_metrics(y_train, y_train_pred, y_train_proba, model_classes)
    test_metrics = calculate_metrics(y_test, y_test_pred, y_test_proba, model_classes)
    argmax_test_metrics = calculate_metrics(y_test, y_test_argmax, y_test_proba, model_classes)

    ci_low, ci_high, ci_n = stratified_bootstrap_auc_ci(
        y_test, y_test_proba, model_classes,
        n_bootstrap=args.bootstrap_iterations,
        random_state=args.random_state,
    )

    cv_auc_scores = manual_cross_validated_auc(best_model, X_train, y_train, cv, model_classes)
    thresholds_df = threshold_diagnostics(y_oof, oof_proba, model_classes, final_thresholds, final_threshold_rule)

    print("Saving evaluation plots...")
    roc_path, roc_class_paths = plot_roc_by_class(y_test, y_test_proba, model_classes, scenario_dir)
    cal_path, cal_class_paths, calibration_metrics_df, calibration_bins_df = plot_calibration_curves(
        y_test, y_test_proba, model_classes, scenario_dir, n_bins=args.calibration_bins
    )

    learning_path = None
    learning_df = pd.DataFrame()
    if not args.skip_learning_curve:
        print("Computing model stability curve...")
        learning_path, learning_df = plot_learning_curve(best_model, X_train, y_train, cv, scenario_dir, n_jobs=args.n_jobs)

    # Post hoc SHAP analyses are performed after model selection and final evaluation.
    shap_paths = []
    shap_importance_df = pd.DataFrame()
    age_threshold_df = pd.DataFrame()
    if args.shap_scope == "all" and not args.skip_shap:
        print("Running post hoc SHAP analysis on the hold-out test set...")
        shap_paths, shap_importance_df, age_threshold_df = run_shap_plots(
            best_model,
            X_train,
            X_test,
            y_test,
            model_classes,
            scenario_dir,
            background_size=args.shap_background_size,
            explain_size=None if args.shap_explain_size is None or args.shap_explain_size < 0 else args.shap_explain_size,
            max_evals=args.shap_max_evals,
            random_state=args.random_state,
            probability_calibrator=probability_calibrator,
        )

    model_path = os.path.join(scenario_dir, f"{scenario_id}_pipeline.joblib")
    threshold_path = os.path.join(scenario_dir, f"{scenario_id}_thresholds.joblib")
    probability_calibration_path = os.path.join(scenario_dir, f"{scenario_id}_probability_calibrator.joblib")
    joblib.dump(best_model, model_path)
    joblib.dump(probability_calibrator, probability_calibration_path)
    joblib.dump({
        "scenario_id": scenario_id,
        "classes": list(model_classes),
        "thresholds": final_thresholds.tolist(),
        "threshold_rule": final_threshold_rule,
        "threshold_rule_key": final_threshold_rule_key,
        "threshold_objective": args.threshold_objective,
        "include_anticoagulant": include_anticoagulant,
        "selection_metric": scenario["metric_definition"],
        "probability_calibration": probability_calibrator,
    }, threshold_path)

    performance_rows = [
        {"Metric": "Test ROCAUC", "Value": test_metrics["ROC-AUC macro OvR"]},
        {"Metric": "Test ROCAUC 95% CI", "Value": f"{ci_low:.4f} to {ci_high:.4f}" if np.isfinite(ci_low) else "NA"},
        {"Metric": "Train ROCAUC", "Value": train_metrics["ROC-AUC macro OvR"]},
        {"Metric": "cross-validation ROCAUC (mean+-SD)", "Value": f"{np.nanmean(cv_auc_scores):.4f} +- {np.nanstd(cv_auc_scores, ddof=1):.4f}"},
        {"Metric": "CV ROC-AUC mean", "Value": float(np.nanmean(cv_auc_scores))},
        {"Metric": "CV ROC-AUC SD", "Value": float(np.nanstd(cv_auc_scores, ddof=1))},
        {"Metric": "Test PR-AUC", "Value": test_metrics["PR-AUC macro"]},
        {"Metric": "Test F1 score", "Value": test_metrics["F1 macro"]},
        {"Metric": "Test F1 weighted", "Value": test_metrics["F1 weighted"]},
        {"Metric": "Accuracy", "Value": test_metrics["Accuracy"]},
        {"Metric": "Test Precision macro", "Value": test_metrics["Precision macro"]},
        {"Metric": "Test Recall macro", "Value": test_metrics["Recall macro"]},
        {"Metric": "Test Specificity macro", "Value": test_metrics["Specificity macro"]},
        {"Metric": "Train F1 macro", "Value": train_metrics["F1 macro"]},
        {"Metric": "Argmax Test F1 macro", "Value": argmax_test_metrics["F1 macro"]},
        {"Metric": "Argmax Accuracy", "Value": argmax_test_metrics["Accuracy"]},
        {"Metric": "Argmax Test ROC-AUC", "Value": argmax_test_metrics["ROC-AUC macro OvR"]},
        {"Metric": "OOF argmax macro-F1", "Value": oof_argmax_f1},
        {"Metric": "OOF argmax accuracy", "Value": oof_argmax_acc},
        {"Metric": "OOF threshold-tuned macro-F1", "Value": oof_threshold_f1},
        {"Metric": "OOF threshold-tuned accuracy", "Value": oof_threshold_acc},
        {"Metric": "OOF threshold objective score", "Value": oof_threshold_obj},
        {"Metric": f"Hyperparameter-search best CV {scenario['refit_metric']}", "Value": search.best_score_},
        {"Metric": "Bootstrap iterations used for Test ROC-AUC CI", "Value": ci_n},
        {"Metric": "CV folds", "Value": n_splits},
        {"Metric": "Final multiclass prediction rule", "Value": final_threshold_rule},
        {"Metric": "Threshold tuning objective", "Value": args.threshold_objective},
        {"Metric": "Troponin included", "Value": "Yes"},
        {"Metric": "Anticoagulant predictor", "Value": "Prespecified" if include_anticoagulant else "Not used"},
        {"Metric": "tPA administered predictor", "Value": "Prespecified"},
        {"Metric": "Selection metric definition", "Value": scenario["metric_definition"]},
        {"Metric": "Probability calibration method", "Value": probability_calibrator.get("method", "identity")},
        {"Metric": "Probability calibration source", "Value": "Training-set out-of-fold predictions only"},
        {"Metric": "OOF calibration objective before calibration", "Value": probability_calibrator.get("oof_before", {}).get("Calibration Objective", np.nan)},
        {"Metric": "OOF calibration objective after calibration", "Value": probability_calibrator.get("oof_selected", {}).get("Calibration Objective", np.nan)},
        {"Metric": "SHAP analysis set", "Value": "Hold-out test set only"},
        {"Metric": "SHAP analysis role", "Value": "Post hoc interpretation only; not used for model selection, hyperparameter tuning, threshold tuning, or performance estimation"},
    ]
    performance_df = pd.DataFrame(performance_rows)

    best_params_df = pd.DataFrame(
        [{"Hyperparameter": k.replace("clf__", "").replace("preprocess__", "preprocess: "), "Value": str(v)}
         for k, v in search.best_params_.items()]
    )
    best_params_df.loc[len(best_params_df)] = ["MLP class-weighting method", clf.weighting_method_]
    best_params_df.loc[len(best_params_df)] = ["MICE imputation max_iter", 30]
    best_params_df.loc[len(best_params_df)] = ["Hyperparameter selection metric", scenario["refit_metric"]]
    best_params_df.loc[len(best_params_df)] = ["Selection metric definition", scenario["metric_definition"]]
    best_params_df.loc[len(best_params_df)] = ["Hyperparameter search", f"RandomizedSearchCV, n_iter={args.n_iter}"]
    best_params_df.loc[len(best_params_df)] = ["Cross-validation design", f"Stratified {n_splits}-fold CV on training set"]
    best_params_df.loc[len(best_params_df)] = ["Threshold tuning", "OOF predictions from training set only; test set not used"]
    best_params_df.loc[len(best_params_df)] = ["Anticoagulant predictor", "Prespecified" if include_anticoagulant else "Not used"]
    best_params_df.loc[len(best_params_df)] = ["Post hoc probability calibration", probability_calibrator.get("method", "identity")]
    best_params_df.loc[len(best_params_df)] = ["Probability calibration fitted on", "Training-set OOF predictions only"]
    best_params_df.loc[len(best_params_df)] = ["tPA administered predictor", "Prespecified"]
    best_params_df.loc[len(best_params_df)] = ["tPA administered encoding", "Nominal categorical; one-hot encoded"]

    class_weights_df = pd.DataFrame({
        "Class": list(clf.class_weight_.keys()),
        "Final Severe Class Weight": list(clf.class_weight_.values()),
    })

    distribution_df = pd.DataFrame({
        "All N": y.value_counts().reindex(model_classes).fillna(0).astype(int),
        "Train N": y_train.value_counts().reindex(model_classes).fillna(0).astype(int),
        "Test N": y_test.value_counts().reindex(model_classes).fillna(0).astype(int),
    }).reset_index().rename(columns={"index": "Class"})
    distribution_df["All %"] = distribution_df["All N"] / distribution_df["All N"].sum() * 100
    distribution_df["Train %"] = distribution_df["Train N"] / distribution_df["Train N"].sum() * 100
    distribution_df["Test %"] = distribution_df["Test N"] / distribution_df["Test N"].sum() * 100

    feature_settings_df = pd.DataFrame({
        "Feature": ALL_FEATURES,
        "Type": [
            "Numeric" if f in NUMERIC_FEATURES else
            "Ordinal" if f in ORDINAL_FEATURES else
            "Binary" if f in BINARY_FEATURES else
            "Nominal" if f in NOMINAL_FEATURES else "Unknown"
            for f in ALL_FEATURES
        ]
    })

    cm_df = pd.DataFrame(
        confusion_matrix(y_test, y_test_pred, labels=model_classes),
        index=[f"Actual: {c}" for c in model_classes],
        columns=[f"Predicted: {c}" for c in model_classes],
    )
    report_df = pd.DataFrame(classification_report(y_test, y_test_pred, labels=model_classes, output_dict=True, zero_division=0)).T

    threshold_compare_df = pd.DataFrame({
        "Rule": ["argmax", "ratio threshold", "subtract threshold", "final selected"],
        "OOF macro-F1": [oof_argmax_f1, oof_thr_f1_ratio, oof_thr_f1_subtract, oof_threshold_f1],
        "OOF accuracy": [oof_argmax_acc, oof_thr_acc_ratio, oof_thr_acc_subtract, oof_threshold_acc],
        "OOF objective": [oof_argmax_obj, oof_obj_ratio, oof_obj_subtract, oof_threshold_obj],
        "Selected": [final_threshold_rule_key == "argmax", final_threshold_rule_key == "ratio", final_threshold_rule_key == "subtract", True],
    })

    plot_files = [roc_path, cal_path]
    plot_files.extend(roc_class_paths)
    plot_files.extend(cal_class_paths)
    if learning_path:
        plot_files.append(learning_path)
    plot_files.extend(shap_paths)
    plot_files_df = pd.DataFrame({"Plot/File": plot_files})

    cv_results_slim = pd.DataFrame(search.cv_results_)
    useful_cols = [c for c in cv_results_slim.columns if c.startswith("mean_test") or c.startswith("std_test") or c.startswith("rank_test") or c == "params"]
    cv_results_slim = cv_results_slim[useful_cols].copy()

    excel_path = os.path.join(scenario_dir, f"{scenario_id}_results.xlsx")
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        performance_df.to_excel(writer, sheet_name="Performance Metrics", index=False)
        best_params_df.to_excel(writer, sheet_name="Best Hyperparameters", index=False)
        class_weights_df.to_excel(writer, sheet_name="Class Weights", index=False)
        thresholds_df.to_excel(writer, sheet_name="Thresholds", index=False)
        threshold_compare_df.to_excel(writer, sheet_name="Threshold Rule Compare", index=False)
        distribution_df.to_excel(writer, sheet_name="Class Distribution", index=False)
        feature_settings_df.to_excel(writer, sheet_name="Feature Settings", index=False)
        calibration_metrics_df.to_excel(writer, sheet_name="Calibration Metrics", index=False)
        calibration_bins_df.to_excel(writer, sheet_name="Calibration Bins", index=False)
        probability_calibration_df.to_excel(writer, sheet_name="Probability Calibration", index=False)
        if not learning_df.empty:
            learning_df.to_excel(writer, sheet_name="Learning Curve", index=False)
        cm_df.to_excel(writer, sheet_name="Confusion Matrix")
        report_df.to_excel(writer, sheet_name="Classification Report")
        cv_results_slim.to_excel(writer, sheet_name="Search CV Results", index=False)
        plot_files_df.to_excel(writer, sheet_name="Saved Files", index=False)
        if not shap_importance_df.empty:
            shap_importance_df.to_excel(writer, sheet_name="SHAP Importance", index=False)
        if not age_threshold_df.empty:
            age_threshold_df.to_excel(writer, sheet_name="Age SHAP Thresholds", index=False)

    summary = {
        "Scenario ID": scenario_id,
        "Scenario": scenario_label,
        "Status": "Completed",
        "Anticoagulant Predictor": "Prespecified" if include_anticoagulant else "Not used",
        "Selection Metric": scenario["metric_definition"],
        "Refit Metric Key": scenario["refit_metric"],
        "Test ROCAUC": test_metrics["ROC-AUC macro OvR"],
        "Test ROCAUC 95% CI": f"{ci_low:.4f} to {ci_high:.4f}" if np.isfinite(ci_low) else "NA",
        "CV ROCAUC Mean": float(np.nanmean(cv_auc_scores)),
        "CV ROCAUC SD": float(np.nanstd(cv_auc_scores, ddof=1)),
        "Test PR-AUC": test_metrics["PR-AUC macro"],
        "Test F1 Macro": test_metrics["F1 macro"],
        "Test F1 Weighted": test_metrics["F1 weighted"],
        "Test Accuracy": test_metrics["Accuracy"],
        "Test Precision Macro": test_metrics["Precision macro"],
        "Test Recall Macro": test_metrics["Recall macro"],
        "Test Specificity Macro": test_metrics["Specificity macro"],
        "OOF Threshold Macro-F1": oof_threshold_f1,
        "OOF Threshold Accuracy": oof_threshold_acc,
        "OOF Threshold Objective": oof_threshold_obj,
        "Best CV Score": search.best_score_,
        "Final Prediction Rule": final_threshold_rule,
        "Probability Calibration": probability_calibrator.get("method", "identity"),
        "Model Path": model_path,
        "Threshold Path": threshold_path,
        "Probability Calibration Path": probability_calibration_path,
        "Excel Path": excel_path,
        "Output Folder": scenario_dir,
    }

    # Store fitted objects for post hoc SHAP interpretation.
    summary["_best_model"] = best_model
    summary["_X_train"] = X_train
    summary["_X_test"] = X_test
    summary["_y_test"] = y_test
    summary["_classes"] = model_classes
    summary["_scenario_dir"] = scenario_dir
    summary["_probability_calibrator"] = probability_calibrator

    print(f"Completed {scenario_id}: Test ROC-AUC={test_metrics['ROC-AUC macro OvR']:.4f}, F1={test_metrics['F1 macro']:.4f}, Accuracy={test_metrics['Accuracy']:.4f}")
    return summary


def run_analysis(args):
    os.makedirs(args.output_dir, exist_ok=True)

    print("Loading data...")
    if not os.path.exists(args.input):
        raise FileNotFoundError(f"Input file not found: {args.input}")
    df_raw = pd.read_excel(args.input, sheet_name=args.sheet)

    summaries = []
    for scenario in SCENARIOS:
        result = run_single_scenario(args, df_raw, scenario)
        summaries.append(result)

    # Keep a reference to the completed final model for optional post hoc interpretation.
    completed = [s for s in summaries if s.get("Status") == "Completed"]
    best_summary = None
    if completed:
        rank_metric_map = {
            "test_rocauc": "Test ROCAUC",
            "test_f1": "Test F1 Macro",
            "test_accuracy": "Test Accuracy",
            "cv_rocauc": "CV ROCAUC Mean",
        }
        rank_col = rank_metric_map.get(args.best_model_metric, "Test ROCAUC")
        best_summary = max(completed, key=lambda s: -np.inf if pd.isna(s.get(rank_col, np.nan)) else float(s.get(rank_col)))

    if args.shap_scope == "best" and not args.skip_shap and best_summary is not None:
        print("\nRunning post hoc SHAP analysis on the hold-out test set...")
        shap_paths, shap_importance_df, age_threshold_df = run_shap_plots(
            best_summary["_best_model"],
            best_summary["_X_train"],
            best_summary["_X_test"],
            best_summary["_y_test"],
            best_summary["_classes"],
            best_summary["_scenario_dir"],
            background_size=args.shap_background_size,
            explain_size=None if args.shap_explain_size is None or args.shap_explain_size < 0 else args.shap_explain_size,
            max_evals=args.shap_max_evals,
            random_state=args.random_state,
            probability_calibrator=best_summary.get("_probability_calibrator"),
        )
        if not shap_importance_df.empty:
            shap_xlsx = os.path.join(best_summary["_scenario_dir"], f"{best_summary['Scenario ID']}_posthoc_shap_importance.xlsx")
            shap_importance_df.to_excel(shap_xlsx, index=False)
        if not age_threshold_df.empty:
            age_xlsx = os.path.join(best_summary["_scenario_dir"], f"{best_summary['Scenario ID']}_age_shap_thresholds.xlsx")
            age_threshold_df.to_excel(age_xlsx, index=False)

    # Remove internal objects before writing master summary.
    clean_summaries = []
    for s in summaries:
        clean = {k: v for k, v in s.items() if not k.startswith("_")}
        clean_summaries.append(clean)

    comparison_df = pd.DataFrame(clean_summaries)
    if not comparison_df.empty and "Scenario ID" in comparison_df.columns:
        comparison_df["Scenario Order"] = comparison_df.apply(scenario_sort_key, axis=1)
        comparison_df = comparison_df.sort_values("Scenario Order").drop(columns=["Scenario Order"])

    completed_df = comparison_df[comparison_df["Status"] == "Completed"].copy() if "Status" in comparison_df.columns else comparison_df.copy()
    best_by_metrics = []
    for metric in ["Test ROCAUC", "Test F1 Macro", "Test Accuracy", "CV ROCAUC Mean", "Test PR-AUC", "Test Specificity Macro"]:
        if metric in completed_df.columns and len(completed_df) > 0:
            idx = pd.to_numeric(completed_df[metric], errors="coerce").idxmax()
            if pd.notna(idx):
                best_by_metrics.append({
                    "Metric": metric,
                    "Best Scenario ID": completed_df.loc[idx, "Scenario ID"],
                    "Best Scenario": completed_df.loc[idx, "Scenario"],
                    "Best Value": completed_df.loc[idx, metric],
                })
    best_by_metrics_df = pd.DataFrame(best_by_metrics)

    master_excel = os.path.join(args.output_dir, "final_round_results.xlsx")
    with pd.ExcelWriter(master_excel, engine="openpyxl") as writer:
        comparison_df.to_excel(writer, sheet_name="Final Results", index=False)
        best_by_metrics_df.to_excel(writer, sheet_name="Best Metric Summary", index=False)
        pd.DataFrame(SCENARIOS).to_excel(writer, sheet_name="Analysis Definition", index=False)

    print("\n" + "=" * 80)
    print("FINAL ROUND SUMMARY")
    print("=" * 80)
    display_cols = [
        "Scenario ID", "Anticoagulant Predictor", "Selection Metric",
        "Test ROCAUC", "CV ROCAUC Mean", "Test F1 Macro", "Test Accuracy",
        "Test Precision Macro", "Test Recall Macro", "Test Specificity Macro", "Status",
    ]
    display_cols = [c for c in display_cols if c in comparison_df.columns]
    print(comparison_df[display_cols].to_string(index=False))
    print("\nBest metric summary:")
    if not best_by_metrics_df.empty:
        print(best_by_metrics_df.to_string(index=False))
    print("\nExcel results:", master_excel)
    print("Output folder:", args.output_dir)
    print("\nPost hoc SHAP interpretation was performed only on the hold-out test set and did not influence model selection, hyperparameter tuning, threshold tuning, or performance estimation.")
    print_final_tuned_hyperparameters(args.output_dir)
    display_final_figures(args.output_dir)


def display_final_figures(output_dir: str) -> None:
    """Display only SHAP interpretation figures at the end of execution.

    ROC, calibration, and learning-curve figures are saved to the output folder
    but are not displayed inline at the end of the run.
    """
    figure_paths = []
    for cls in TARGET_CLASSES:
        figure_paths.append(os.path.join(output_dir, f"shap_beeswarm_{safe_filename(cls)}.png"))
    for cls in TARGET_CLASSES:
        figure_paths.append(os.path.join(output_dir, f"shap_age_threshold_{safe_filename(cls)}.png"))

    existing_paths = [p for p in figure_paths if os.path.exists(p)]
    if not existing_paths:
        return

    print("\nDisplaying final SHAP figures...")
    try:
        from IPython.display import Image as IPyImage, display
        for p in existing_paths:
            print(os.path.basename(p))
            display(IPyImage(filename=p))
    except Exception:
        try:
            from PIL import Image
            import matplotlib.pyplot as plt
            for p in existing_paths:
                img = Image.open(p)
                plt.figure(figsize=(10, 6))
                plt.imshow(img)
                plt.axis("off")
                plt.title(os.path.basename(p))
                plt.show()
        except Exception:
            for p in existing_paths:
                print(f"Figure saved: {p}")


def print_final_tuned_hyperparameters(output_dir: str) -> None:
    """Print the final selected hyperparameter settings at the end of execution."""
    excel_path = os.path.join(output_dir, "final_macro_f1_model_results.xlsx")
    if not os.path.exists(excel_path):
        return

    try:
        params_df = pd.read_excel(excel_path, sheet_name="Best Hyperparameters")
    except Exception:
        return

    if params_df.empty or "Hyperparameter" not in params_df.columns or "Value" not in params_df.columns:
        return

    print("\n" + "=" * 80)
    print("FINAL TUNED HYPERPARAMETERS")
    print("=" * 80)

    for _, row in params_df.iterrows():
        name = str(row["Hyperparameter"])
        value = str(row["Value"])
        print(f"{name}: {value}")


# -----------------------------------------------------------------------------
# CLI
# -----------------------------------------------------------------------------

def parse_args():
    parser = argparse.ArgumentParser(description="Final MLP analysis with macro-F1 hyperparameter selection and post hoc SHAP interpretation")
    parser.add_argument("--input", default="/content/StrokeData_ML_dataset.xlsx", help="Input Excel file path")
    parser.add_argument("--sheet", default=0, help="Excel sheet name or index")
    parser.add_argument("--output-dir", default="/content/Final Round", help="Output folder")
    parser.add_argument("--test-size", type=float, default=0.20, help="Held-out test fraction")
    parser.add_argument("--random-state", type=int, default=42, help="Random seed")
    parser.add_argument("--bootstrap-iterations", type=int, default=1000, help="Bootstrap iterations for 95% CI")
    parser.add_argument("--n-iter", type=int, default=300, help="Number of randomized hyperparameter candidates")
    parser.add_argument("--threshold-iter", type=int, default=60000, help="Iterations for threshold random search")
    parser.add_argument("--threshold-objective", choices=["macro_f1", "macro_f1_accuracy", "balanced"], default="macro_f1_accuracy", help="Threshold tuning objective")
    parser.add_argument("--calibration-bins", type=int, default=10, help="Number of bins for ECE and calibration curve")
    parser.add_argument("--probability-calibration", choices=["auto", "none", "temperature", "vector_scaling"], default="auto", help="Post hoc probability calibration fitted using training-set OOF predictions only")
    parser.add_argument("--probability-calibration-l2", type=float, default=0.02, help="L2 shrinkage for vector-scaling probability calibration")
    parser.add_argument("--skip-learning-curve", action="store_true", help="Skip learning curves to save runtime")
    parser.add_argument("--skip-shap", action="store_true", help="Skip post hoc SHAP plots")
    parser.add_argument("--shap-scope", choices=["none", "best", "all"], default="all", help="Run post hoc SHAP on the final fitted model")
    parser.add_argument("--best-model-metric", choices=["test_rocauc", "test_f1", "test_accuracy", "cv_rocauc"], default="test_rocauc", help="Compatibility option; a single final model is evaluated")
    parser.add_argument("--shap-background-size", type=int, default=120, help="SHAP background sample size")
    parser.add_argument("--shap-explain-size", type=int, default=-1, help="Number of hold-out test samples to explain by SHAP; use -1 for the full test set")
    parser.add_argument("--shap-max-evals", type=int, default=900, help="SHAP permutation max_evals")
    parser.add_argument("--n-jobs", type=int, default=-1, help="Parallel jobs for search/CV")
    parser.add_argument("--verbose", type=int, default=1, help="RandomizedSearchCV verbosity")
    parser.add_argument("--require-anticoagulant", action="store_true", help="Compatibility option; the final analysis requires Anticoagulant")
    args, _unknown = parser.parse_known_args()

    try:
        args.sheet = int(args.sheet)
    except Exception:
        pass
    if args.shap_scope == "none":
        args.skip_shap = True
    return args


if __name__ == "__main__":
    run_analysis(parse_args())